# КЭШ

In [1]:
from typing import Any, Optional
import time

class CacheItem:
    def __init__(self, value: Any, ttl: Optional[int] = None):
        self.value = value
        self.created_at = time.time()
        self.ttl = ttl

    def is_expired(self) -> bool:
        if self.ttl is None:
            return False
        return (time.time() - self.created_at) > self.ttl

In [2]:
class InMemoryCache:
    """
    Простой in-memory cache с TTL.
    """

    def __init__(self):
        self._store: dict[str, CacheItem] = {}

    def exists(self, key: str) -> bool:
        item = self._store.get(key)
        if not item:
            return False

        if item.is_expired():
            self.invalidate(key)
            return False

        return True

    def get(self, key: str) -> Any:
        if not self.exists(key):
            return None
        return self._store[key].value

    def set(self, key: str, value: Any, ttl: Optional[int] = None):
        self._store[key] = CacheItem(value=value, ttl=ttl)

    def invalidate(self, key: str):
        self._store.pop(key, None)

    def clear(self):
        self._store.clear()

In [3]:
import json
import hashlib

def make_cache_key(prefix: str, payload: Any) -> str:
    """
    Делает стабильный hash-ключ из произвольного payload.
    """
    payload_str = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    digest = hashlib.sha256(payload_str.encode("utf-8")).hexdigest()
    return f"{prefix}:{digest}"

In [4]:
# Smoke test
cache = InMemoryCache()

key = make_cache_key("test", {"a": 1, "b": 2})
cache.set(key, "hello", ttl=2)

print(cache.get(key))   # hello
time.sleep(3)
print(cache.get(key))   # None (expired)

hello
None


# Инициализация локальной LLM

In [5]:
!pip install -U langchain-huggingface bitsandbytes transformers

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# -------------------------
# Загрузка модели
# -------------------------

def load_qwen_pipeline(
    temperature: float = 0.3,
    max_new_tokens: int = 512,
):
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True
    )
        # квантование
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
        )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        repetition_penalty=1.1,
        return_full_text=False,
    )

    return HuggingFacePipeline(pipeline=pipe)

# -------------------------
# LLM wrapper
# -------------------------

class LocalLLM:
    """
    Единая точка доступа к LLM.
    """

    def __init__(self, llm_pipeline):
        self.llm = llm_pipeline
        self.cache = cache

    def generate(self, prompt: str, ttl: int = 3600) -> str:
        llm_key = make_cache_key("llm", prompt)

        if self.cache.exists(llm_key):
          return self.cache.get(llm_key)

        response = self.llm.invoke(prompt)

        if isinstance(response, dict):
            text = response.get("text", "")
        else:
            text = str(response)

        text = text.strip()

        self.cache.set(llm_key, text, ttl=ttl)
        return text


# -------------------------
# Инициализация
# -------------------------

def init_llm():
    pipe = load_qwen_pipeline()
    return LocalLLM(pipe)
# -------------------------
# Smoke test
# -------------------------

llm = init_llm()
print(llm.generate("Объясни, что такое среднее значение."))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda:0


Среднее значение - это мера центральной тенденции в наборе данных, которая представляет собой сумму всех значений в наборе данных, деленную на количество этих значений. Это один из наиболее распространенных способов описания "среднего" или "обычного" значения в наборе данных.

Пример: Если у нас есть набор чисел {1, 2, 3, 4, 5}, то среднее значение будет равно (1 + 2 + 3 + 4 + 5) / 5 = 3. 

Среднее значение может быть использовано для анализа различных типов данных, таких как возраст людей, стоимость товаров и т.д. Однако стоит помнить, что среднее значение может быть не всегда лучшим представителем всей выборки, особенно если данные имеют выбросы или распределены неравномерно. В таких случаях могут быть более полезны другие меры центральной тенденции, такие как медиана или мода.


# RAG-инфраструктура для PDF

In [7]:
!pip install -U sentence-transformers faiss-cpu pdfplumber rank-bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.1 MB/s eta 0:00:00


In [8]:
# Чтение PDF
import pdfplumber
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def read_pdf_text(path: str) -> str:
    """Читает весь текст PDF и возвращает как одну строку."""
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n".join(pages)

#  Извлечение строк схемы (колонка + описание)
def extract_schema_rows(pdf_text: str) -> list[tuple[str, str]]:
    """
    Возвращает список кортежей (column_name, description)
    на основе строк PDF.
    """
    rows = []
    pattern = re.compile(r"^([a-zA-Z_0-9]+)\s+(.*)$")  # имя колонки + описание
    for line in pdf_text.splitlines():
        match = pattern.match(line.strip())
        if match:
            col, desc = match.groups()
            rows.append((col, desc.strip()))
    return rows


In [9]:
# Построение semantic chunks (1 строка = 1 чанк)
def build_schema_chunks(rows: list[tuple[str, str]]) -> list[str]:
    chunks = []
    for col, desc in rows:
        chunk = f"column: {col}\ndescription: {desc}"
        chunks.append(chunk)
    return chunks

In [10]:
# ============= ГИБРИДНЫЙ RAG =============
class HybridPDFRAG:
    """
    Комбинирует BM25 (лексический поиск) + Semantic embeddings для лучшей релевантности.
    """
    def __init__(self, chunks: list[str], embedding_model: str = "paraphrase-multilingual-MiniLM-L12-v2"):
        self.chunks = chunks

        # BM25 индекс
        tokenized_chunks = [chunk.lower().split() for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized_chunks)

        # Semantic embeddings
        self.embedder = SentenceTransformer(embedding_model)
        embeddings = self.embedder.encode(
            chunks,
            convert_to_numpy=True,
            show_progress_bar=True
        )
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)

    def retrieve(self, question: str, k: int = 3, bm25_weight: float = 0.3, semantic_weight:  float = 0.7) -> list[str]:
        """
        Гибридный поиск: BM25 + Semantic.
        Args:
            question: вопрос пользователя
            k: количество результатов
            bm25_weight: вес BM25 поиска (0-1)
            semantic_weight: вес семантического поиска (0-1)
        Returns:
            top-k чанков, отранжированных по гибридной метрике
        """

        # ===== BM25 ПОИСК =====
        question_tokens = question.lower().split()
        bm25_scores = self.bm25.get_scores(question_tokens)

        # Нормализуем BM25 scores в диапазон [0, 1]
        bm25_scores_normalized = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)

        # ===== SEMANTIC ПОИСК =====
        question_embedding = self.embedder.encode([question])
        distances, indices = self.index.search(question_embedding, len(self.chunks))

        # Переводим расстояния в similarty scores (меньше расстояние = выше score)
        # Используем нормализованные расстояния
        semantic_scores = np.zeros(len(self.chunks))
        distances = distances[0]
        max_distance = distances.max() + 1e-8
        semantic_scores[indices[0]] = 1 - (distances / max_distance)

        # ===== ГИБРИДНАЯ РАНЖИРОВКА =====
        hybrid_scores = (bm25_weight * bm25_scores_normalized +
                        semantic_weight * semantic_scores)

        # Берём top-k по гибридному скору
        top_indices = np.argsort(-hybrid_scores)[:k]

        # Возвращаем с сортировкой по скорам
        results = [
            (self.chunks[idx], hybrid_scores[idx])
            for idx in top_indices
        ]
        results.sort(key=lambda x: x[1], reverse=True)

        return [chunk for chunk, score in results]


In [11]:
# Простейший вопрос (entry-point)
def ask_rag(question: str, rag: HybridPDFRAG):
    print(f"\nВОПРОС:\n{question}\n")
    retrieved_chunks = rag.retrieve(question)
    print("RETRIEVED CHUNKS:\n")
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"--- CHUNK {i} ---")
        print(chunk, "\n")

* PDF файл надо сначала вручную в колаб загрузить


In [12]:
# ============= ИНИЦИАЛИЗАЦИЯ =============
PDF_PATH = "/content/ТБ-Отели и а_б.pdf"  # путь к PDF

pdf_text = read_pdf_text(PDF_PATH)
rows = extract_schema_rows(pdf_text)
schema_dict = {col: desc for col, desc in rows}  # для прямого доступа
chunks = build_schema_chunks(rows)

# Создаём гибридный RAG
rag = HybridPDFRAG(chunks)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
# ============= SMOKE TEST =============
question = "Что означает promo_code_discount_amt?"
ask_rag(question, rag)

# Тест с другим вопросом
question2 = "какие поля описывают скидку?"
ask_rag(question2, rag)



ВОПРОС:
Что означает promo_code_discount_amt?

RETRIEVED CHUNKS:

--- CHUNK 1 ---
column: promo_code_discount_amt
description: Размер скидки по промокоду 

--- CHUNK 2 ---
column: nominal_price_rub_amt
description: Сумма заказа в рублях 

--- CHUNK 3 ---
column: created_dttm
description: Дата создания заказа 


ВОПРОС:
какие поля описывают скидку?

RETRIEVED CHUNKS:

--- CHUNK 1 ---
column: nominal_price_rub_amt
description: Сумма заказа в рублях 

--- CHUNK 2 ---
column: promo_code_discount_amt
description: Размер скидки по промокоду 

--- CHUNK 3 ---
column: bundle_nm
description: Подписка, применившаяся к оплате Pro/Premium 



# Обработка csv и Executor

In [32]:
# Обработка csv
import pandas as pd
from typing import Dict

#CSV_PATH = "/content/hotel_tickets_data_test_sample.csv"

CSV_PATH = "/content/hotel_tickets_data.csv"

def load_and_prepare_df(csv_path: str, schema_dict: Dict[str, str]) -> pd.DataFrame:
    df = pd.read_csv(csv_path, sep=";", encoding="utf-8")

    print(f"Загружено строк: {len(df)}, колонок: {len(df.columns)}")
    print("Колонки в данных:", list(df.columns))

    # ---------- 1. Корректная проверка схемы ----------
    schema_cols = {col for col in schema_dict if "_" in col or "age" in col}
    df_cols = set(df.columns)

    missing_cols = schema_cols - df_cols
    extra_cols = df_cols - schema_cols

    if missing_cols:
        print(f"\nВНИМАНИЕ: В CSV отсутствуют колонки из PDF-схемы:\n{missing_cols}")

    if extra_cols:
        print(f"\nДополнительные колонки в CSV (не описаны в PDF):\n{extra_cols}")

    # ---------- 2. Приведение типов ----------
    for col in df.columns:
        desc = schema_dict.get(col, "").lower()

        # --- даты ---
        if (
            col.endswith("_dt")
            or col.endswith("_dttm")
            or "дата" in desc
            or "date" in desc
        ):
            df[col] = pd.to_datetime(df[col], errors="coerce")

        # --- числовые ---
        elif (
            col.endswith("_amt")
            or "сумма" in desc
            or "amount" in desc
            or "price" in col.lower()
            or "discount" in col.lower()
        ):
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", "."),
                errors="coerce"
            )

        # --- флаги ---
        elif col.endswith("_flg"):
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", "."),
                errors="coerce"
            ).astype("Int8")

        # --- категориальные ---
        elif col.endswith("_cd") or col.endswith("_nm"):
            df[col] = df[col].astype("category")

    print("\nОсновные статистики:")
    print(df.describe(include="all").to_string())

    return df

In [33]:
df = load_and_prepare_df(CSV_PATH, schema_dict)

/tmp/ipython-input-2068633947.py:10: DtypeWarning: Columns (31,32,33,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, sep=";", encoding="utf-8")


Загружено строк: 835938, колонок: 56
Колонки в данных: ['order_online_payment_flg', 'account_rk', 'client_rk', 'order_rk', 'loyalty_program_type_nm', 'bundle_nm', 'order_type_cd', 'order_status_cd', 'party_first_order_dt', 'party_first_order_type_dt', 'free_cancel_booking_dttm', 'created_dttm', 'cancel_dttm', 'book_start_dttm', 'local_book_start_dttm', 'book_end_dttm', 'hotel_country', 'hotel_city', 'avia_dep_city', 'avia_arr_city', 'promo_code_discount_amt', 'loyalty_accrual_rub_amt', 'nominal_price_eur_amt', 'nominal_price_rub_amt', 'order_item_cnt', 'month_beginning_balance_rub', 'monthly_income_amt', 'suppress_email_flg', 'suppress_call_flg', 'bounce_cd', 'last_sms_success_flg', 'call_contact_6m_flg', 'call_contact_3m_flg', 'call_contact_1m_flg', 'good_email_address_flg', 'bad_email_address_flg', 'email_valid_flg', 'children_cnt', 'age', 'age_type_cd', 'parent_meeting_region_nm', 'delivery_region_category_cd', 'lvn_city_nm', 'lvn_state_nm', 'time_zone_delta_tm', 'time_zone_cd', 'la

# Interpreter и ClarificationHandler

In [34]:
"""
Простой InterpreterNode с логикой обнаружения неоднозначностей на Python уровне.
"""

import json
import re
from typing import Dict, Any, List


class InterpreterNode:
    """
    InterpreterNode: понимает вопрос, выделяет интенции, обнаруживает неоднозначности.
    """

    def __init__(self, llm, rag, schema_dict, df):
        self.llm = llm
        self.rag = rag
        self.schema_dict = schema_dict
        self.df = df

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """Выполняет интерпретацию вопроса"""
        question = state.get("question", "")

        print(f"\n🔍 InterpreterNode: {question}")

        # === STEP 1: RAG retrieval ===
        print("  📚 Ищу релевантные описания колонок...")
        retrieved_chunks = self.rag.retrieve(question, k=5)

        schema_context = "\n".join(retrieved_chunks)
        print(f"  ✓ Найдено {len(retrieved_chunks)} релевантных описаний")

        # === STEP 2: LLM анализирует вопрос ===
        print("  🤖 Анализирую вопрос с LLM...")

        prompt = f"""Ты аналитический ассистент. Проанализируй вопрос пользователя и определи требуемую метрику.

ВОПРОС: {question}

ДОСТУПНЫЕ КОЛОНКИ:
{schema_context}

ИНСТРУКЦИИ:
- Если вопрос про "цена", "стоимость", "дорог", "дешев" → метрика: mean (средняя)
- Если вопрос про "активн", "популяр", "часто", "сколько" → метрика: count (количество)
- Если вопрос про "всего", "итого", "сумма" → метрика: sum (сумма)
- Для тренда (менялась, динамика) → обычно mean или sum
- Для ранжирования (какой лучший, худший) → обычно count или sum

Ответь ТОЛЬКО валидный JSON:
{{
    "main_metric": "count|sum|mean|max|min",
    "group_by": "имя_колонки_или_null",
    "filters": [],
    "required_columns": ["список_необходимых_колонок"],
    "confidence": 0.0_до_1.0
}}

Проверь, что JSON валидный! Используй двойные кавычки везде!"""

        response = self.llm.generate(prompt)

        # === STEP 2.5: Ищем первый валидный JSON в ответе ===
        intentions = self._extract_first_valid_json(response)

        if intentions is None:
            print(f"  ❌ Не найден ни один валидный JSON, берём fallback")
            intentions = {
                "main_metric": "count",
                "group_by": None,
                "filters": [],
                "required_columns": [],
                "confidence": 0.5
            }
        else:
            print(f"  ✓ JSON успешно распарсен")

        # === STEP 3: Добавляем автофильтр по типу продукта ===
        intentions = self._add_product_filter(question, intentions)

        # === STEP 4: Обнаруживаем неоднозначности (на Python) ===
        print("  🔎 Проверяю неоднозначности...")

        ambiguities = self._detect_ambiguities(
            question,
            intentions,
            retrieved_chunks
        )

        if ambiguities:
            print(f"  ⚠️ Обнаружены {len(ambiguities)} неоднозначность/неоднозначности:")
            for amb in ambiguities:
                print(f"     - {amb['issue']}")

            state["clarification_needed"] = True
            state["ambiguities"] = ambiguities
            state["clarification_question"] = self._format_clarification_question(ambiguities)
        else:
            state["clarification_needed"] = False
            state["ambiguities"] = []
            state["clarification_question"] = None

        state["intentions"] = intentions
        state["schema_context"] = schema_context
        state["retrieved_chunks"] = retrieved_chunks

        return state

    def _extract_first_valid_json(self, response: str) -> Dict[str, Any]:
        """
        Извлекает первый валидный JSON из ответа LLM.
        LLM может выдать много текста, нужно найти JSON объект.
        """
        # Ищем все JSON объекты в ответе (от { до })
        json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'

        matches = re.finditer(json_pattern, response, re.DOTALL)

        for match in matches:
            json_str = match.group(0)
            try:
                obj = json.loads(json_str)

                # Валидируем, что это именно то, что нам нужно
                if self._validate_intentions_object(obj):
                    print(f"    ✓ Найден валидный JSON на позиции {match.start()}")
                    return obj
            except json.JSONDecodeError:
                # Этот JSON невалиден, пробуем следующий
                continue

        # Если не нашли ни одного валидного JSON
        return None

    def _validate_intentions_object(self, obj: Any) -> bool:
        """
        Проверяет, что объект имеет нужную структуру для intentions.
        """
        if not isinstance(obj, dict):
            return False

        # Проверяем обязательные поля
        required_fields = ["main_metric", "group_by", "filters", "required_columns", "confidence"]

        for field in required_fields:
            if field not in obj:
                return False

        # Проверяем типы
        if not isinstance(obj["main_metric"], str):
            return False

        if obj["group_by"] is not None and not isinstance(obj["group_by"], str):
            return False

        if not isinstance(obj["filters"], list):
            return False

        if not isinstance(obj["required_columns"], list):
            return False

        if not isinstance(obj["confidence"], (int, float)):
            return False

        # Проверяем допустимые значения метрики
        valid_metrics = ["count", "sum", "mean", "max", "min"]
        if obj["main_metric"] not in valid_metrics:
            return False

        return True

    def _add_product_filter(self, question: str, intentions: Dict[str, Any]) -> Dict[str, Any]:
        """
        Добавляет автоматический фильтр по типу продукта.
        """
        question_lower = question.lower()

        # === Определяем о чём вопрос ===
        is_discount = any(word in question_lower for word in ["скидк", "промокод", "discount", "promo"])
        is_price = any(word in question_lower for word in ["цена", "стоимость", "price", "cost"])
        is_aviate = "авиа" in question_lower or "билет" in question_lower
        is_hotel = "отел" in question_lower

        # === Если вопрос про скидку ===
        if is_discount:
            # Добавляем колонку скидки в required_columns
            if "required_columns" not in intentions:
                intentions["required_columns"] = []
            if "promo_code_discount_amt" not in intentions["required_columns"]:
                intentions["required_columns"].append("promo_code_discount_amt")
            print(f"  ✓ Добавлена колонка: promo_code_discount_amt")

        # === Фильтр по типу продукта (если упоминается) ===
        if is_aviate:
            filter_obj = {
                "column": "order_type_cd",
                "value": "AIR",
                "op": "=="
            }
            if filter_obj not in intentions.get("filters", []):
                if "filters" not in intentions:
                    intentions["filters"] = []
                intentions["filters"].append(filter_obj)
                print(f"  ✓ Добавлен автофильтр: order_type_cd == 'AIR'")

        elif is_hotel:
            filter_obj = {
                "column": "order_type_cd",
                "value": "HOT",
                "op": "=="
            }
            if filter_obj not in intentions.get("filters", []):
                if "filters" not in intentions:
                    intentions["filters"] = []
                intentions["filters"].append(filter_obj)
                print(f"  ✓ Добавлен автофильтр: order_type_cd == 'HOT'")

        return intentions

    def _detect_ambiguities(
        self,
        question: str,
        intentions: Dict[str, Any],
        retrieved_chunks: List[str]
    ) -> List[Dict[str, Any]]:
        """
        Обнаруживает неоднозначности на основе:
        1. Ключевых слов в вопросе
        2. Найденных колонок в retrieved_chunks
        3. Данных в DataFrame
        """

        ambiguities = []
        question_lower = question.lower()
        schema_str = "\n".join(retrieved_chunks).lower()

        # === Проверка 1: Валюта ===
        if "цена" in question_lower or "стоимость" in question_lower or "сумма" in question_lower:
            # Ищем валютные колонки в retrieved chunks
            price_columns = [
                col for col in self.df.columns
                if "price" in col.lower() or "amt" in col.lower()
            ]

            # Считаем варианты валют
            rub_cols = [col for col in price_columns if "rub" in col.lower()]
            eur_cols = [col for col in price_columns if "eur" in col.lower()]

            # Если больше одной валютной опции — неоднозначность
            if len(rub_cols) > 0 and len(eur_cols) > 0:
                ambiguities.append({
                    "issue": "Указана цена, но не ясна валюта",
                    "type": "currency",
                    "options": {
                        "RUB": rub_cols,
                        "EUR": eur_cols
                    },
                    "question": "В какой валюте показать цену: рубли (RUB) или евро (EUR)?"
                })

        # === Проверка 2: Временной период ===
        if "менялась" in question_lower or "тренд" in question_lower or "динамика" in question_lower:
            # Проверяем, явно ли указан год/период
            has_explicit_period = any(
                year in question_lower
                for year in ["2023", "2022", "2024", "2021", "2020","2019","2018"]
            ) or any(
                period in question_lower
                for period in ["год", "месяц", "неделя", "день", "квартал"]
            )

            if not has_explicit_period:
                ambiguities.append({
                    "issue": "Вопрос про динамику, но период не уточнен",
                    "type": "period",
                    "question": "За какой период показать динамику?"
                })

        # === Проверка 3: Уровень детализации ===
        if any(word in question_lower for word in ["менялась", "тренд", "динамика"]):
            # Ищем временные колонки
            date_cols = [
                col for col in self.df.columns
                if "date" in col.lower() or "dttm" in col.lower() or "time" in col.lower()
            ]

            # Если много вариантов дат — неоднозначность
            if len(date_cols) > 1:
                ambiguities.append({
                    "issue": "Несколько вариантов временных колонок",
                    "type": "date_column",
                    "options": date_cols,
                    "question": f"Какую дату использовать для анализа? [{', '.join(date_cols[:3])}...]"
                })

        return ambiguities

    def _format_clarification_question(self, ambiguities: List[Dict[str, Any]]) -> str:
        """Форматирует вопросы уточнения для пользователя"""

        questions = []

        for i, amb in enumerate(ambiguities, 1):
            q = f"{i}. {amb['question']}"

            # Если есть опции — добавляем их
            if "options" in amb and isinstance(amb["options"], dict):
                options_str = ", ".join(amb["options"].keys())
                q += f" [{options_str}]"
            elif "options" in amb and isinstance(amb["options"], list):
                options_str = ", ".join(amb["options"][:3])
                q += f" [{options_str}...]"

            questions.append(q)

        return "\n".join(questions)

In [35]:
"""
Обработчик уточнений от пользователя.
Собирает ответы, валидирует их и возвращает структурированные данные для Planner.
"""

from typing import Dict, Any, List, Optional


class ClarificationHandler:
    """
    Обрабатывает уточнения от пользователя и интегрирует их в state.
    """

    def __init__(self):
        self.clarifications = {}

    def ask_clarification(
        self,
        question: str,
        ambiguity_type: str,
        options: Optional[List[str]] = None
    ) -> str:
        """
        Выводит вопрос и собирает ответ пользователя.

        Args:
            question: вопрос для пользователя
            ambiguity_type: тип неоднозначности (currency, date_column, etc)
            options: список возможных вариантов

        Returns:
            ответ пользователя
        """

        print(f"\n❓ {question}")

        if options:
            print(f"   Доступные варианты:")
            for i, opt in enumerate(options, 1):
                print(f"   {i}. {opt}")

            print(f"\n   Выбери номер или введи значение:")
        else:
            print(f"\n   Введи значение:")

        # Получаем ввод пользователя
        user_input = input("   → ").strip()

        # Валидируем (если это номер опции)
        if options and user_input.isdigit():
            idx = int(user_input) - 1
            if 0 <= idx < len(options):
                answer = options[idx]
            else:
                print(f"   ❌ Неверный номер. Выбираю первый вариант: {options[0]}")
                answer = options[0]
        else:
            answer = user_input

        print(f"   ✓ Выбран: {answer}")

        self.clarifications[ambiguity_type] = answer
        return answer

    def collect_all_clarifications(
        self,
        ambiguities: List[Dict[str, Any]]
    ) -> Dict[str, Any]:
        """
        Собирает все уточнения от пользователя.

        Args:
            ambiguities: список неоднозначностей из InterpreterNode

        Returns:
            словарь с ответами {type: answer}
        """

        print("\n" + "=" * 80)
        print("📋 ТРЕБУЮТСЯ УТОЧНЕНИЯ")
        print("=" * 80)

        self.clarifications = {}

        for amb in ambiguities:
            amb_type = amb.get("type")
            question = amb.get("question")

            # Подготавливаем опции
            options = None
            if "options" in amb and isinstance(amb["options"], dict):
                # Для словаря (типа currency)
                options = list(amb["options"].keys())
            elif "options" in amb and isinstance(amb["options"], list):
                # Для списка (типа date_column)
                options = amb["options"]

            # Спрашиваем пользователя
            self.ask_clarification(question, amb_type, options)

        print("\n" + "=" * 80)
        print("✅ ВСЕ УТОЧНЕНИЯ СОБРАНЫ")
        print("=" * 80)

        return self.clarifications

    def apply_clarifications_to_state(
        self,
        state: Dict[str, Any],
        clarifications: Dict[str, Any]
    ) -> Dict[str, Any]:
        """
        Применяет уточнения к state для передачи в Planner.

        Args:
            state: текущий state из Interpreter
            clarifications: ответы пользователя

        Returns:
            обновленный state с уточнениями
        """

        print("\n📝 Применяю уточнения к плану...")

        # Копируем intentions
        intentions = state.get("intentions", {})

        # === Обработка уточнения по валюте ===
        if "currency" in clarifications:
            currency = clarifications["currency"]
            print(f"  ✓ Валюта: {currency}")

            # Найдём нужную колонку для цены
            if currency == "RUB":
                # Выбираем RUB колонку (обычно nominal_price_rub_amt)
                price_col = "nominal_price_rub_amt"
            elif currency == "EUR":
                price_col = "nominal_price_eur_amt"
            else:
                price_col = currency

            # Добавляем в required_columns если её там нет
            if "required_columns" not in intentions:
                intentions["required_columns"] = []

            if price_col not in intentions["required_columns"]:
                intentions["required_columns"].append(price_col)

            # Сохраняем выбор
            state["selected_currency"] = currency
            state["price_column"] = price_col

        # === Обработка уточнения по дате ===
        if "date_column" in clarifications:
            date_col = clarifications["date_column"]
            print(f"  ✓ Дата: {date_col}")

            # Обновляем group_by если это был вопрос про динамику
            if intentions.get("main_metric") in ("mean", "sum", "count"):
                # Проверяем, есть ли в вопросе слова про динамику
                question = state.get("question", "").lower()
                if any(word in question for word in ["менялась", "тренд", "динамика"]):
                    intentions["group_by"] = date_col
                    print(f"  ✓ Обновлён group_by: {date_col}")

            # Добавляем в required_columns
            if "required_columns" not in intentions:
                intentions["required_columns"] = []

            if date_col not in intentions["required_columns"]:
                intentions["required_columns"].append(date_col)

            state["selected_date_column"] = date_col

        # Обновляем intentions в state
        state["intentions"] = intentions
        state["clarifications_applied"] = True

        print(f"  ✓ Updated required_columns: {intentions.get('required_columns', [])}")

        return state

# Planer

In [36]:
"""
PlannerNode - формирует абстрактный план на основе intentions.
"""
from typing import Dict, Any, List
import json

class PlannerNode:
    """
    Формирует высокоуровневый план анализа.
    Не привязан к конкретным колонкам — только логика.
    Преобразует intentions → abstract plan с шагами анализа
    """

    def __init__(self, llm: 'LocalLLM'):
        self.llm = llm

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Input: state с intentions из Interpreter + уточнениями от пользователя
        Output: state с абстрактным планом для QueryBuilder
        """
        question = state.get("question", "")
        intentions = state.get("intentions", {})

        print(f"\n📋 PlannerNode")
        print(f"   Question: {question}")

        # === Проверяем, есть ли интенции ===
        if not intentions or not intentions.get("required_columns"):
            print("  ❌ Нет intentions, план не может быть составлен")
            state["plan"] = None
            state["plan_valid"] = False
            return state

        # === Определяем тип анализа ===
        question_lower = question.lower()

        # Флаги для определения типа анализа
        is_trend = any(word in question_lower for word in ["менялась", "тренд", "динамика", "как изменилась"])
        is_ranking = any(word in question_lower for word in ["лучш", "худш", "больш", "мень", "наиб", "популяр", "топ", "активн"])
        is_aggregation = any(word in question_lower for word in ["всего", "итого", "средн", "средний", "сумма"])
        is_comparison = any(word in question_lower for word in ["сравни", "отличи", "разниц"])

        print(f"  📊 Анализ типа вопроса:")
        print(f"     - Тренд: {is_trend}")
        print(f"     - Ранжирование: {is_ranking}")
        print(f"     - Агрегация: {is_aggregation}")
        print(f"     - Сравнение: {is_comparison}")

        # === Формируем план ===
        if is_trend:
            plan = self._build_trend_plan(intentions)
        elif is_ranking:
            plan = self._build_ranking_plan(intentions)
        elif is_comparison:
            plan = self._build_comparison_plan(intentions)
        elif is_aggregation:
            plan = self._build_aggregation_plan(intentions)
        else:
            # Default — агрегация
            plan = self._build_aggregation_plan(intentions)

        print(f"  ✓ План составлен с {len(plan['steps'])} шагами")

        state["plan"] = plan
        state["plan_valid"] = True
        state["analysis_type"] = self._determine_analysis_type(is_trend, is_ranking, is_comparison, is_aggregation)

        return state

    def _determine_analysis_type(self, is_trend, is_ranking, is_comparison, is_aggregation) -> str:
        """Определяет основной тип анализа"""
        if is_trend:
            return "trend"
        elif is_ranking:
            return "ranking"
        elif is_comparison:
            return "comparison"
        elif is_aggregation:
            return "aggregation"
        else:
            return "general"

    def _get_metric_column(self, intentions: Dict[str, Any]) -> str:
        """
        Находит подходящую колонку для метрики.
        Ищет в required_columns числовую/денежную колонку.
        """
        required_columns = intentions.get("required_columns", [])

        # === ПРИОРИТЕТ ПОИСКА ===
        # 1. Ищем скидку (promo_code_discount_amt)
        for col in required_columns:
            if "discount" in col.lower() or "promo" in col.lower():
                return col

        # 2. Ищем price колонку
        for col in required_columns:
            if "price" in col.lower() or "amt" in col.lower():
                return col

        # 3. Проверяем все доступные колонки в датасете
        for col in self.df.columns:
            if "discount" in col.lower():
                return col

        for col in self.df.columns:
            if "price" in col.lower() or "amt" in col.lower():
                return col

        # Fallback
        return "nominal_price_rub_amt"

    def _build_trend_plan(self, intentions: Dict[str, Any]) -> Dict[str, Any]:
        """
        План для вопросов про динамику/тренд.
        Например: "Как менялась цена авиабилетов в 2023 году?"
        """
        print("    → Составляю план анализа ТРЕНДА")

        plan = {
            "type": "trend",
            "description": "Анализ динамики метрики за период",
            "steps": []
        }

        # Шаг 1: Фильтрация (если нужна)
        filters = intentions.get("filters", [])
        if filters:
            plan["steps"].append({
                "step": "filter",
                "type": "conditional",
                "conditions": filters,
                "description": f"Применяем {len(filters)} фильтр(ов)"
            })

        # Шаг 2: Агрегация по времени
        group_by = intentions.get("group_by")
        main_metric = intentions.get("main_metric", "mean")
        metric_column = self._get_metric_column(intentions)

        if group_by:
            plan["steps"].append({
                "step": "aggregate",
                "type": "time_series",
                "group_by": group_by,
                "metrics": [
                    {
                        "type": main_metric,
                        "column": metric_column,
                        "name": f"{main_metric}_price"
                    }
                ],
                "description": f"Агрегируем {main_metric}({metric_column}) по {group_by}"
            })

        # Шаг 3: Сортировка по времени
        plan["steps"].append({
            "step": "sort",
            "by": group_by or "date",
            "order": "asc",
            "description": "Сортируем по времени (возрастанию)"
        })

        return plan

    def _build_ranking_plan(self, intentions: Dict[str, Any]) -> Dict[str, Any]:
        """
        План для вопросов про ранжирование.
        Например: "Какая программа лояльности наиболее активно покупает?"
        """
        print("    → Составляю план РАНЖИРОВАНИЯ")

        plan = {
            "type": "ranking",
            "description": "Ранжирование по метрике",
            "steps": []
        }

        # Шаг 1: Фильтрация (если нужна)
        filters = intentions.get("filters", [])
        if filters:
            plan["steps"].append({
                "step": "filter",
                "conditions": filters,
                "description": f"Применяем {len(filters)} фильтр(ов)"
            })

        # Шаг 2: Агрегация и группировка
        group_by = intentions.get("group_by")
        main_metric = intentions.get("main_metric", "count")
        metric_column = self._get_metric_column(intentions) if main_metric != "count" else None

        if group_by:
            metric_obj = {
                "type": main_metric,
                "name": f"{main_metric}_total"
            }
            if metric_column:
                metric_obj["column"] = metric_column

            plan["steps"].append({
                "step": "aggregate",
                "type": "groupby",
                "group_by": group_by,
                "metrics": [metric_obj],
                "description": f"Группируем по {group_by}, считаем {main_metric}"
            })

        # Шаг 3: Сортировка по метрике (по убыванию)
        plan["steps"].append({
            "step": "sort",
            "by": f"{main_metric}_total",
            "order": "desc",
            "description": "Сортируем по метрике (по убыванию)"
        })

        # Шаг 4: Лимит топ-N
        plan["steps"].append({
            "step": "limit",
            "rows": 10,
            "description": "Берём топ-10 результатов"
        })

        return plan

    def _build_comparison_plan(self, intentions: Dict[str, Any]) -> Dict[str, Any]:
        """
        План для вопросов про сравнение.
        Например: "Сравни цены в разных авиакомпаниях"
        """
        print("    → Составляю план СРАВНЕНИЯ")

        plan = {
            "type": "comparison",
            "description": "Сравнение метрик между группами",
            "steps": []
        }

        # Шаг 1: Фильтрация
        filters = intentions.get("filters", [])
        if filters:
            plan["steps"].append({
                "step": "filter",
                "conditions": filters,
                "description": f"Применяем {len(filters)} фильтр(ов)"
            })

        # Шаг 2: Агрегация по группам
        group_by = intentions.get("group_by")
        main_metric = intentions.get("main_metric", "mean")
        metric_column = self._get_metric_column(intentions)

        if group_by:
            plan["steps"].append({
                "step": "aggregate",
                "type": "comparison",
                "group_by": group_by,
                "metrics": [
                    {
                        "type": main_metric,
                        "column": metric_column,
                        "name": f"{main_metric}_value"
                    },
                    {
                        "type": "count",
                        "name": "count_items"
                    }
                ],
                "description": f"Агрегируем {main_metric} для каждой группы"
            })

        # Шаг 3: Сортировка
        plan["steps"].append({
            "step": "sort",
            "by": f"{main_metric}_value",
            "order": "desc",
            "description": "Сортируем по метрике"
        })

        return plan

    def _build_aggregation_plan(self, intentions: Dict[str, Any]) -> Dict[str, Any]:
        """
        План для простых вопросов про агрегацию.
        Например: "Какой средний чек на авиабилет?"
        """
        print("    → Составляю план АГРЕГАЦИИ")

        plan = {
            "type": "aggregation",
            "description": "Простая агрегация данных",
            "steps": []
        }

        # Шаг 1: Фильтрация (если нужна)
        filters = intentions.get("filters", [])
        if filters:
            plan["steps"].append({
                "step": "filter",
                "conditions": filters,
                "description": f"Применяем {len(filters)} фильтр(ов)"
            })

        # Шаг 2: Агрегация
        main_metric = intentions.get("main_metric", "count")
        group_by = intentions.get("group_by")
        metric_column = self._get_metric_column(intentions) if main_metric != "count" else None

        metric_obj = {
            "type": main_metric,
            "name": f"{main_metric}_result"
        }
        if metric_column:
            metric_obj["column"] = metric_column

        plan["steps"].append({
            "step": "aggregate",
            "type": "simple" if not group_by else "groupby",
            "group_by": group_by,
            "metrics": [metric_obj],
            "description": f"Вычисляем {main_metric}" + (f" по {group_by}" if group_by else "")
        })

        # Шаг 3: Сортировка (если есть группировка)
        if group_by:
            plan["steps"].append({
                "step": "sort",
                "by": f"{main_metric}_result",
                "order": "desc",
                "description": "Сортируем по результату"
            })

            # Шаг 4: Лимит
            plan["steps"].append({
                "step": "limit",
                "rows": 10,
                "description": "Берём топ-10"
            })

        return plan

# QueryBuilder

In [37]:
"""
QueryBuilderNode - преобразует абстрактный план в JSON query для SimpleAnalyticsExecutor
"""
from typing import Dict, Any, List
import re

class QueryBuilderNode:
    """
    Преобразует план из Planner → JSON query для SimpleAnalyticsExecutor
    """

    def __init__(self, df):
        self.df = df

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Input: state с plan из Planner
        Output: state с query готовым для executor
        """

        plan = state.get("plan", {})
        question = state.get("question", "")

        print(f"\n🔨 QueryBuilderNode")
        print(f"   Question: {question}")

        if not plan or not plan.get("steps"):
            print("  ❌ План пуст, query не может быть построен")
            state["query"] = None
            state["query_valid"] = False
            return state

        # Инициализируем query структуру
        query = {
            "filters": [],
            "group_by": None,
            "metrics": [],
            "sort_by": None,
            "sort_order": "asc",
            "limit": None
        }

        # Обходим все шаги плана
        print(f"  📝 Обрабатываю {len(plan['steps'])} шагов плана...")

        for step in plan.get("steps", []):
            step_type = step.get("step")

            if step_type == "filter":
                query = self._process_filter_step(step, query, question)

            elif step_type == "aggregate":
                query = self._process_aggregate_step(step, query)

            elif step_type == "sort":
                query = self._process_sort_step(step, query)

            elif step_type == "limit":
                query = self._process_limit_step(step, query)

            elif step_type == "compute_derivatives":
                # Пропускаем — это не поддерживается SimpleAnalyticsExecutor
                print(f"    ⚠️ Шаг '{step_type}' пропущен (не поддерживается)")
                continue

            else:
                print(f"    ⚠️ Неизвестный шаг: {step_type}")

        # Добавляем фильтр по году, если он упоминается в вопросе
        query = self._add_year_filter(question, query)

        # Валидируем query
        is_valid = self._validate_query(query)

        state["query"] = query
        state["query_valid"] = is_valid

        if is_valid:
            print(f"  ✅ Query построен успешно")
            self._print_query_summary(query)
        else:
            print(f"  ❌ Query невалиден")

        return state

    def _process_filter_step(self, step: Dict[str, Any], query: Dict[str, Any], question: str) -> Dict[str, Any]:
        """Обрабатывает шаг filter"""
        conditions = step.get("conditions", [])

        for condition in conditions:
            # Пропускаем фильтры с op="year_equals" — обработаем их отдельно
            if condition.get("op") == "year_equals":
                continue

            query["filters"].append(condition)

        if query["filters"]:
            print(f"    ✓ Фильтры добавлены: {len(query['filters'])} шт")

        return query

    def _process_aggregate_step(self, step: Dict[str, Any], query: Dict[str, Any]) -> Dict[str, Any]:
        """Обрабатывает шаг aggregate"""

        query["group_by"] = step.get("group_by")

        # Обрабатываем метрики
        metrics_list = step.get("metrics", [])

        if isinstance(metrics_list, list) and len(metrics_list) > 0:
            # Если это просто список типов метрик
            if isinstance(metrics_list[0], str):
                # Преобразуем в полную структуру
                for metric_type in metrics_list:
                    metric_obj = self._build_metric_object(metric_type, query["group_by"])
                    query["metrics"].append(metric_obj)
            else:
                # Уже полные метрики
                query["metrics"] = metrics_list

        print(f"    ✓ Агрегация: group_by='{query['group_by']}', метрики={len(query['metrics'])} шт")

        return query

    def _process_sort_step(self, step: Dict[str, Any], query: Dict[str, Any]) -> Dict[str, Any]:
        """Обрабатывает шаг sort"""

        sort_by = step.get("by")
        sort_order = step.get("order", "asc")

        query["sort_by"] = sort_by
        query["sort_order"] = sort_order

        print(f"    ✓ Сортировка: by='{sort_by}', order='{sort_order}'")

        return query

    def _process_limit_step(self, step: Dict[str, Any], query: Dict[str, Any]) -> Dict[str, Any]:
        """Обрабатывает шаг limit"""

        limit = step.get("rows")
        query["limit"] = limit

        print(f"    ✓ Лимит: {limit} строк")

        return query

    def _add_year_filter(self, question: str, query: Dict[str, Any]) -> Dict[str, Any]:
        """
        Добавляет фильтр по году если он упоминается в вопросе.
        Работает сложнее, т.к. нужно фильтровать по диапазону дат.
        """

        # Ищем год в вопросе
        year_match = re.search(r'\b(202[0-9])\b', question)
        if not year_match:
            return query

        year = int(year_match.group(1))

        # Ищем дату, по которой фильтровать
        # Обычно это created_dttm для авиабилетов
        date_col = query.get("group_by")
        if not date_col or "dttm" not in date_col.lower():
            # Если group_by не дата, ищем в filters
            date_col = None
            for filt in query["filters"]:
                if "dttm" in filt.get("column", "").lower():
                    date_col = filt["column"]
                    break

        if not date_col:
            date_col = "created_dttm"  # default

        # Добавляем два фильтра: >= 1 янв и < 1 янв след года
        filter_year_start = {
            "column": date_col,
            "value": f"{year}-01-01",
            "op": ">="
        }

        filter_year_end = {
            "column": date_col,
            "value": f"{year + 1}-01-01",
            "op": "<"
        }

        query["filters"].append(filter_year_start)
        query["filters"].append(filter_year_end)

        print(f"    ✓ Добавлен фильтр по году {year} на колонке '{date_col}'")

        return query

    def _build_metric_object(self, metric_type: str, group_by: str = None) -> Dict[str, Any]:
        """
        Конструирует полный объект метрики из типа.
        Гарантирует, что каждая метрика имеет требуемую колонку.
        """

        # Определяем колонку для метрики в зависимости от типа
        if metric_type == "count":
            # Count может работать без конкретной колонки
            column = None
            name = "count_total"

        elif metric_type in ("mean", "sum", "min", "max"):
            # Для числовых метрик по цене
            column = "nominal_price_rub_amt"  # default
            name = f"{metric_type}_price"

        else:
            # Fallback
            column = None
            name = f"{metric_type}_result"

        return {
            "type": metric_type,
            "column": column,
            "name": name
        }

    def _validate_query(self, query: Dict[str, Any]) -> bool:
        """
        Валидирует query перед отправкой в executor.
        Проверяет:
        - Структура фильтров корректна
        - Если есть метрики с mean/sum/min/max, то у них есть колонка
        - Если есть группировка, то есть метрики
        """

        # Проверяем обязательные поля
        if not isinstance(query.get("filters"), list):
            print("    ❌ Ошибка: фильтры должны быть списком")
            return False

        # Если есть группировка, нужны метрики
        if query.get("group_by") and not query.get("metrics"):
            print("    ❌ Ошибка: если указан group_by, нужны метрики")
            return False

        # Если есть метрики, они должны быть валидны
        for metric in query.get("metrics", []):
            if not isinstance(metric, dict):
                print(f"    ❌ Ошибка: метрика должна быть dict, получено {type(metric)}")
                return False

            if "type" not in metric:
                print(f"    ❌ Ошибка: в метрике отсутствует 'type'")
                return False

            metric_type = metric.get("type")

            # Проверяем, что числовые метрики имеют колонку
            if metric_type in ("mean", "sum", "min", "max"):
                if not metric.get("column"):
                    print(f"    ❌ Ошибка: метрика '{metric_type}' требует колонку, но колонка не указана")
                    return False

            # Проверяем, что колонка существует в датасете (если указана)
            if metric.get("column") and metric.get("column") not in self.df.columns:
                print(f"    ❌ Ошибка: колонка '{metric.get('column')}' не существует в датасете")
                return False

        # Проверяем, что колонки для фильтров существуют
        for filt in query.get("filters", []):
            col = filt.get("column")
            if col and col not in self.df.columns:
                print(f"    ❌ Ошибка: колонка для фильтра '{col}' не существует в датасете")
                return False

        # Проверяем, что group_by колонка существует
        if query.get("group_by") and query.get("group_by") not in self.df.columns:
            print(f"    ❌ Ошибка: колонка group_by '{query.get('group_by')}' не существует в датасете")
            return False

        return True

    def _print_query_summary(self, query: Dict[str, Any]):
        """Выводит краткую информацию о query"""

        print(f"\n  📊 Query Summary:")
        print(f"     Фильтры: {len(query['filters'])} шт")
        if query['filters']:
            for filt in query['filters']:
                print(f"       - {filt['column']} {filt['op']} {filt['value']}")

        if query['group_by']:
            print(f"     Group by: {query['group_by']}")
            print(f"     Метрики: {len(query['metrics'])} шт")
            for metric in query['metrics']:
                col_str = f" ({metric.get('column', 'N/A')})" if metric.get('column') else ""
                print(f"       - {metric['type']}{col_str} → {metric.get('name', 'unknown')}")

        if query['sort_by']:
            print(f"     Sort: {query['sort_by']} ({query['sort_order']})")

        if query['limit']:
            print(f"     Limit: {query['limit']} rows")

# SimpleAnalyticsExecutor

In [38]:
"""
SimpleAnalyticsExecutor — упрощённая версия для простых аналитических вопросов.
Поддерживает только:
- Фильтрацию (filter)
- Агрегацию (aggregation)
- Сравнение (comparison)
- Базовую сегментацию (segmentation)

НЕ поддерживает:
- Производные признаки
- Извлечение признаков (feature extraction)
- Сложные артефакты
"""

import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional, Union
import hashlib
import json
from datetime import datetime
import time


class SimpleAnalyticsExecutor:
    """
    Минимальный исполнитель для простых аналитических вопросов.
    Поддерживает: filter → aggregation → comparison/segmentation
    """

    def __init__(
        self,
        df: pd.DataFrame,
        schema_dict: Dict[str, str] = None,
        cache: Optional['InMemoryCache'] = None,
    ):
        self.df = self._prepare_dataframe(df.copy())
        self.schema_dict = schema_dict or {}
        self.cache = cache

    def _prepare_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """Минимальная предобработка: даты и числа"""
        for col in df.columns:
            # Пробуем преобразовать в дату
            if col.lower().endswith(('_dt', '_date', '_dttm')) or 'дата' in col.lower():
                try:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                except:
                    pass
            # Пробуем преобразовать в число
            elif col.lower().endswith(('_amt', '_price', '_cost')) or any(x in col.lower() for x in ['цена', 'сумма', 'стоимость']):
                try:
                    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')
                except:
                    pass
        return df

    def execute_simple_query(self, query: Dict[str, Any]) -> Dict[str, Any]:
        """
        Выполняет простой аналитический запрос.
        """
        start_time = time.time()

        # Проверяем кэш
        cache_key = self._make_cache_key(query)
        if self.cache and self.cache.exists(cache_key):
            print("📦 Результат найден в кэше")
            result = self.cache.get(cache_key)
            result["from_cache"] = True
            return result

        print(f"▶️ Выполняем запрос...")

        # Начинаем с полного датасета
        df_work = self.df.copy()

        # === STEP 1: ФИЛЬТРАЦИЯ ===
        filters = query.get("filters", [])
        if filters:
            print(f"  🔍 Применяем {len(filters)} фильтр(ов)...")
            for filt in filters:
                col = filt.get("column")
                val = filt.get("value")
                op = filt.get("op", "==")

                if op == "==":
                    df_work = df_work[df_work[col] == val]
                elif op == "!=":
                    df_work = df_work[df_work[col] != val]
                elif op == ">":
                    df_work = df_work[df_work[col] > val]
                elif op == "<":
                    df_work = df_work[df_work[col] < val]
                elif op == ">=":
                    df_work = df_work[df_work[col] >= val]
                elif op == "<=":
                    df_work = df_work[df_work[col] <= val]
                elif op == "in":
                    df_work = df_work[df_work[col].isin(val if isinstance(val, list) else [val])]

                print(f"    ✓ '{col}' {op} {val} → {len(df_work)} строк")

        # === STEP 2: АГРЕГАЦИЯ ===
        metrics = query.get("metrics", [])
        group_by = query.get("group_by")

        if metrics and group_by:
            # === С ГРУППИРОВКОЙ ===
            print(f"  ∑ Агрегируем по '{group_by}'...")

            # Если group_by это дата — группируем по дате (без времени)
            if "dttm" in group_by.lower() or "date" in group_by.lower():
                print(f"    📅 Обрезаю время в '{group_by}'...")
                df_work[f"{group_by}_date"] = pd.to_datetime(df_work[group_by]).dt.date
                group_by_col = f"{group_by}_date"
            else:
                group_by_col = group_by

            agg_dict = {}
            for metric in metrics:
                metric_type = metric.get("type")
                metric_name = metric.get("name", f"{metric_type}_result")

                if metric_type == "count":
                    agg_dict[metric_name] = (metric.get("column", df_work.columns[0]), "count")
                elif metric_type == "sum":
                    col = metric.get("column")
                    agg_dict[metric_name] = (col, "sum")
                elif metric_type == "mean":
                    col = metric.get("column")
                    agg_dict[metric_name] = (col, "mean")
                elif metric_type == "min":
                    col = metric.get("column")
                    agg_dict[metric_name] = (col, "min")
                elif metric_type == "max":
                    col = metric.get("column")
                    agg_dict[metric_name] = (col, "max")

            if agg_dict:
                result_df = df_work.groupby(group_by_col).agg(**agg_dict).reset_index()
                # Переименовываем дата колонку обратно
                if group_by_col != group_by:
                    result_df.rename(columns={group_by_col: group_by}, inplace=True)
            else:
                result_df = df_work.groupby(group_by_col).size().reset_index(name="count")

        elif metrics and not group_by:
            # === БЕЗ ГРУППИРОВКИ (простая агрегация) ===
            print(f"  ∑ Выполняю простую агрегацию без группировки...")

            result_data = {}
            for metric in metrics:
                metric_type = metric.get("type")
                metric_name = metric.get("name", f"{metric_type}_result")
                col = metric.get("column")

                if metric_type == "count":
                    # Для скидок - считаем только ненулевые
                    if "discount" in col.lower():
                        result_data[metric_name] = len(df_work[df_work[col] > 0])
                        print(f"    ✓ Подсчитано строк со скидкой > 0: {result_data[metric_name]}")
                    else:
                        result_data[metric_name] = len(df_work)

                elif metric_type == "sum" and col:
                    # Пропускаем NaN и нулевые при суммировании скидок
                    if "discount" in col.lower():
                        result_data[metric_name] = float(df_work[df_work[col] > 0][col].sum())
                    else:
                        result_data[metric_name] = float(df_work[col].sum())

                elif metric_type == "mean" and col:
                    # Для средней скидки - только ненулевые значения
                    if "discount" in col.lower():
                        discount_values = df_work[df_work[col] > 0][col]
                        result_data[metric_name] = float(discount_values.mean())
                        result_data[f"{metric_name}_count"] = len(discount_values)
                        print(f"    ✓ Средняя скидка (только > 0): {result_data[metric_name]:.2f}")
                        print(f"    ✓ Количество записей со скидкой: {result_data[f'{metric_name}_count']}")
                    else:
                        result_data[metric_name] = float(df_work[col].mean())

                elif metric_type == "min" and col:
                    result_data[metric_name] = float(df_work[col].min())

                elif metric_type == "max" and col:
                    result_data[metric_name] = float(df_work[col].max())

            # Создаём dataframe с одной строкой
            result_df = pd.DataFrame([result_data])
            print(f"    ✓ Результаты агрегации: {result_data}")

        else:
            # === ПРОСТО ВЫБОРКА ===
            print(f"  ℹ️ Нет метрик, возвращаю первые строки...")
            result_df = df_work.head(10)

        # === STEP 3: СОРТИРОВКА И ЛИМИТ ===
        sort_by = query.get("sort_by")
        sort_order = query.get("sort_order", "asc")
        limit = query.get("limit", None)

        if sort_by and sort_by in result_df.columns:
            ascending = (sort_order == "asc")
            result_df = result_df.sort_values(by=sort_by, ascending=ascending)
            print(f"  📊 Сортируем по '{sort_by}' ({sort_order})")

        if limit:
            result_df = result_df.head(limit)
            print(f"  ⚙️ Ограничиваем до {limit} строк")

        # === ПОДГОТОВКА РЕЗУЛЬТАТА ===
        execution_time = time.time() - start_time

        # Конвертируем для JSON
        result_data = result_df.to_dict('records')

        result = {
            "success": True,
            "data": result_data,
            "row_count": len(result_df),
            "column_count": len(result_df.columns),
            "columns": list(result_df.columns),
            "execution_time": execution_time,
            "from_cache": False,
            "timestamp": datetime.now().isoformat()
        }

        # Кэшируем результат
        if self.cache:
            try:
                self.cache.set(cache_key, result, ttl=3600)
            except:
                pass

        print(f"✅ Готово за {execution_time:.3f}s ({len(result_df)} строк)")
        return result

    def _make_cache_key(self, query: Dict[str, Any]) -> str:
        """Генерирует ключ кэша"""
        query_str = json.dumps(query, sort_keys=True, default=str)
        query_hash = hashlib.sha256(query_str.encode()).hexdigest()[:16]

        data_hash = hashlib.sha256(
            f"{len(self.df)}_{','.join(sorted(self.df.columns))}".encode()
        ).hexdigest()[:16]

        return make_cache_key("simple_query", f"{query_hash}_{data_hash}")

# ============================================================================
# ИНТЕГРАЦИЯ С СИСТЕМОЙ
# ============================================================================

def init_simple_executor(df: pd.DataFrame, schema_dict: Dict[str, str], cache) -> SimpleAnalyticsExecutor:
    """Инициализирует простой исполнитель"""
    return SimpleAnalyticsExecutor(df, schema_dict, cache)



# Executor узел

In [39]:
"""
ExecutorNode - выполняет query через SimpleAnalyticsExecutor и возвращает результаты
"""
from typing import Dict, Any
import time

class ExecutorNode:
    """
    Выполняет JSON query через SimpleAnalyticsExecutor
    """

    def __init__(self, executor: 'SimpleAnalyticsExecutor'):
        self.executor = executor

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Input: state с query из QueryBuilder
        Output: state с results
        """

        query = state.get("query")
        question = state.get("question", "")

        print(f"\n⚙️ ExecutorNode")
        print(f"   Question: {question}")

        if not query:
            print("  ❌ Query пуст, выполнение невозможно")
            state["execution_result"] = None
            state["execution_success"] = False
            return state

        if not state.get("query_valid"):
            print("  ❌ Query невалиден, выполнение невозможно")
            state["execution_result"] = None
            state["execution_success"] = False
            return state

        # === Выполняем query ===
        print(f"  🔄 Выполняю query через executor...")

        try:
            start_time = time.time()

            result = self.executor.execute_simple_query(query)

            execution_time = time.time() - start_time

            # === Добавляем метаданные ===
            result["executor_execution_time"] = execution_time
            result["query_used"] = query

            state["execution_result"] = result
            state["execution_success"] = True

            # === Выводим информацию ===
            self._print_execution_summary(result)

        except Exception as e:
            print(f"  ❌ Ошибка при выполнении: {str(e)}")
            import traceback
            traceback.print_exc()

            state["execution_result"] = None
            state["execution_success"] = False
            state["execution_error"] = str(e)

        return state

    def _print_execution_summary(self, result: Dict[str, Any]):
        """Выводит краткую информацию о результатах"""

        print(f"\n  📊 Execution Summary:")
        print(f"     Success: {result.get('success', False)}")
        print(f"     Rows returned: {result.get('row_count', 0)}")
        print(f"     Columns: {result.get('column_count', 0)}")
        print(f"     Execution time: {result.get('execution_time', 0):.3f}s")
        print(f"     From cache: {result.get('from_cache', False)}")

        if result.get('data'):
            print(f"\n     Первые 5 строк результата:")
            for i, row in enumerate(result['data'][:5], 1):
                print(f"       {i}. {row}")

            if len(result['data']) > 5:
                print(f"       ... и ещё {len(result['data']) - 5} строк")


# ============================================================================
# ИНТЕГРАЦИЯ
# ============================================================================

def init_executor_node(df, schema_dict, cache) -> ExecutorNode:
    """Инициализирует ExecutorNode"""
    executor = SimpleAnalyticsExecutor(df, schema_dict, cache)
    return ExecutorNode(executor)

# Analyzer

In [40]:
"""
AnalyzerNode - анализирует результаты и добавляет аналитическое объяснение
"""
from typing import Dict, Any
import pandas as pd
import numpy as np


class AnalyzerNode:
    """
    Анализирует результаты из Executor и добавляет аналитику.
    """

    def __init__(self):
        pass

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Input: state с execution_result из Executor
        Output: state с analysis
        """

        result = state.get("execution_result")
        question = state.get("question", "")
        query = state.get("query", {})
        analysis_type = state.get("analysis_type", "general")

        print(f"\n📊 AnalyzerNode")
        print(f"   Question: {question}")
        print(f"   Analysis type: {analysis_type}")

        if not result or not result.get("success"):
            print("  ❌ Нет результатов для анализа")
            state["analysis"] = None
            state["analysis_success"] = False
            return state

        try:
            # Конвертируем результаты в DataFrame для анализа
            df_result = pd.DataFrame(result.get("data", []))

            if df_result.empty:
                print("  ❌ Результаты пусты")
                state["analysis"] = None
                state["analysis_success"] = False
                return state

            print(f"  ✓ Получено {len(df_result)} строк для анализа")

            # === Выполняем анализ ===
            analysis = self._analyze_results(
                df_result,
                query,
                analysis_type,
                question
            )

            state["analysis"] = analysis
            state["analysis_success"] = True

            # === Выводим информацию ===
            self._print_analysis_summary(analysis)

        except Exception as e:
            print(f"  ❌ Ошибка при анализе: {str(e)}")
            import traceback
            traceback.print_exc()

            state["analysis"] = None
            state["analysis_success"] = False
            state["analysis_error"] = str(e)

        return state

    def _analyze_results(
        self,
        df: pd.DataFrame,
        query: Dict[str, Any],
        analysis_type: str,
        question: str
    ) -> Dict[str, Any]:
        """
        Выполняет анализ результатов в зависимости от типа.
        """

        analysis = {
            "type": analysis_type,
            "total_rows": len(df),
            "columns": list(df.columns),
            "timestamp": pd.Timestamp.now().isoformat(),
            "insights": []
        }

        # === Определяем числовые и текстовые колонки ===
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()

        # === Дополнительно ищем date колонки (datetime.date) ===
        for col in df.columns:
            if df[col].dtype == 'object' and len(df) > 0:
                try:
                    first_val = df[col].iloc[0]
                    # Проверяем, это дата?
                    if hasattr(first_val, 'year') and hasattr(first_val, 'month'):
                        if 'dttm' in col.lower() or 'date' in col.lower():
                            datetime_cols.append(col)
                except:
                    pass

        print(f"    📈 Числовые колонки: {numeric_cols}")
        print(f"    📅 Дата-время колонки: {datetime_cols}")

        # === Базовая статистика по числовым колонкам ===
        if numeric_cols:
            analysis["statistics"] = {}
            for col in numeric_cols:
                stats = {
                    "min": float(df[col].min()),
                    "max": float(df[col].max()),
                    "mean": float(df[col].mean()),
                    "median": float(df[col].median()),
                    "std": float(df[col].std()),
                    "count_non_null": int(df[col].count())
                }
                analysis["statistics"][col] = stats
                print(f"    ✓ Статистика для {col}: min={stats['min']:.2f}, max={stats['max']:.2f}, mean={stats['mean']:.2f}")

        # === Анализ в зависимости от типа ===
        if analysis_type == "trend":
            analysis = self._analyze_trend(df, analysis, numeric_cols, datetime_cols)

        elif analysis_type == "ranking":
            analysis = self._analyze_ranking(df, analysis, numeric_cols)

        elif analysis_type == "comparison":
            analysis = self._analyze_comparison(df, analysis, numeric_cols)

        elif analysis_type == "aggregation":
            analysis = self._analyze_aggregation(df, analysis, numeric_cols)

        return analysis

    def _analyze_trend(
        self,
        df: pd.DataFrame,
        analysis: Dict[str, Any],
        numeric_cols: list,
        datetime_cols: list
    ) -> Dict[str, Any]:
        """Анализирует тренды"""

        print("    🔍 Анализирую тренд...")

        if numeric_cols and datetime_cols:
            metric_col = numeric_cols[0]
            date_col = datetime_cols[0]

            # Сортируем по дате
            df_sorted = df.sort_values(date_col)

            # Вычисляем тренд
            values = df_sorted[metric_col].values
            dates = df_sorted[date_col].values

            if len(values) > 1:
                # Изменение от начала к концу
                change = values[-1] - values[0]
                percent_change = (change / values[0] * 100) if values[0] != 0 else 0

                # Направление тренда
                trend_direction = "📈 вверх" if change > 0 else "📉 вниз" if change < 0 else "➡️ стабильно"

                analysis["trend"] = {
                    "metric_column": metric_col,
                    "date_column": date_col,
                    "start_value": float(values[0]),
                    "end_value": float(values[-1]),
                    "start_date": str(dates[0]),
                    "end_date": str(dates[-1]),
                    "change": float(change),
                    "percent_change": float(percent_change),
                    "trend_direction": trend_direction,
                    "data_points": len(values)
                }

                insight = f"Тренд {trend_direction}: {metric_col} изменился с {values[0]:.2f} до {values[-1]:.2f} ({percent_change:+.1f}%) за {len(values)} дней"
                analysis["insights"].append(insight)
                print(f"    ✓ {insight}")

                # Ищем максимум и минимум
                max_idx = np.argmax(values)
                min_idx = np.argmin(values)

                analysis["trend"]["max_value"] = float(values[max_idx])
                analysis["trend"]["min_value"] = float(values[min_idx])
                analysis["trend"]["max_date"] = str(dates[max_idx])
                analysis["trend"]["min_date"] = str(dates[min_idx])
                analysis["trend"]["volatility"] = float(np.std(values))

                # Дополнительный insight про волатильность
                if analysis["trend"]["volatility"] > np.mean(values) * 0.5:
                    insight_vol = f"⚠️ Высокая волатильность: стандартное отклонение = {analysis['trend']['volatility']:.2f}"
                    analysis["insights"].append(insight_vol)
                    print(f"    ⚠️ {insight_vol}")

        return analysis

    def _analyze_ranking(
        self,
        df: pd.DataFrame,
        analysis: Dict[str, Any],
        numeric_cols: list
    ) -> Dict[str, Any]:
        """Анализирует ранжирование"""

        print("    🏆 Анализирую ранжирование...")

        if numeric_cols:
            metric_col = numeric_cols[0]

            # Топ элемент
            top_row = df.iloc[0]
            top_value = float(top_row[metric_col])

            # Получаем название (если есть group_by колонка)
            group_by_cols = [c for c in df.columns if c != metric_col]
            top_name = str(top_row[group_by_cols[0]]) if group_by_cols else "Топ"

            analysis["ranking"] = {
                "metric_column": metric_col,
                "top_item": top_name,
                "top_value": top_value,
                "total_items": len(df),
                "top_percentage": float((top_value / df[metric_col].sum() * 100) if df[metric_col].sum() != 0 else 0)
            }

            insight = f"🏆 Лидер: {top_name} с значением {top_value:.2f} ({analysis['ranking']['top_percentage']:.1f}% от общего)"
            analysis["insights"].append(insight)
            print(f"    ✓ {insight}")

        return analysis

    def _analyze_comparison(
        self,
        df: pd.DataFrame,
        analysis: Dict[str, Any],
        numeric_cols: list
    ) -> Dict[str, Any]:
        """Анализирует сравнение"""

        print("    ⚖️ Анализирую сравнение...")

        if numeric_cols:
            metric_col = numeric_cols[0]

            # Вычисляем разброс
            min_val = float(df[metric_col].min())
            max_val = float(df[metric_col].max())
            spread = max_val - min_val
            spread_percent = (spread / min_val * 100) if min_val != 0 else 0

            analysis["comparison"] = {
                "metric_column": metric_col,
                "min_value": min_val,
                "max_value": max_val,
                "spread": spread,
                "spread_percent": float(spread_percent),
                "items_count": len(df)
            }

            insight = f"Разброс значений: от {min_val:.2f} до {max_val:.2f} ({spread_percent:.1f}%)"
            analysis["insights"].append(insight)
            print(f"    ✓ {insight}")

        return analysis

    def _analyze_aggregation(self, df: pd.DataFrame, analysis: Dict[str, Any], numeric_cols: list) -> Dict[str, Any]:
        """Анализирует агрегацию"""
        print("    ∑ Анализирую агрегацию...")

        if numeric_cols:
            metric_col = numeric_cols[0]
            value = float(df[metric_col].iloc[0]) if len(df) > 0 else 0

            # Ищем count колонку
            count_col = None
            for col in df.columns:
                if "count" in col.lower():
                    count_col = col
                    break

            count_value = None
            if count_col and len(df) > 0:
                count_value = int(df[count_col].iloc[0])

            analysis["aggregation"] = {
                "metric_column": metric_col,
                "value": value,
                "total_rows": len(df),
                "count_rows": count_value
            }

            insight = f"Итоговое значение {metric_col}: {value:.2f}"
            analysis["insights"].append(insight)
            print(f"    ✓ {insight}")

            if count_value:
                count_insight = f"Количество записей с этим значением: {count_value}"
                analysis["insights"].append(count_insight)
                print(f"    ✓ {count_insight}")

        return analysis

    def _print_analysis_summary(self, analysis: Dict[str, Any]):
        """Выводит краткую информацию об анализе"""

        print(f"\n  📈 Analysis Summary:")
        print(f"     Type: {analysis.get('type')}")
        print(f"     Total rows: {analysis.get('total_rows')}")
        print(f"     Columns: {analysis.get('columns')}")

        if analysis.get("insights"):
            print(f"\n     Insights ({len(analysis['insights'])}):")
            for i, insight in enumerate(analysis["insights"], 1):
                print(f"       {i}. {insight}")

        if analysis.get("statistics"):
            print(f"\n     Statistics:")
            for col, stats in analysis["statistics"].items():
                print(f"       {col}:")
                print(f"         min={stats['min']:.2f}, max={stats['max']:.2f}, mean={stats['mean']:.2f}")

        if analysis.get("trend"):
            trend = analysis["trend"]
            print(f"\n     Trend Details:")
            print(f"       Start: {trend['start_value']:.2f} ({trend['start_date']})")
            print(f"       End: {trend['end_value']:.2f} ({trend['end_date']})")
            print(f"       Max: {trend['max_value']:.2f} ({trend['max_date']})")
            print(f"       Min: {trend['min_value']:.2f} ({trend['min_date']})")
            print(f"       Volatility: {trend['volatility']:.2f}")

# Answer узел

In [41]:
"""
AnswerGeneratorNode - генерирует финальный текстовый ответ на основе анализа
"""
from typing import Dict, Any


class AnswerGeneratorNode:
    """
    Генерирует естественный текстовый ответ на вопрос пользователя.
    """

    def __init__(self, llm=None):
        self.llm = llm

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Input: state с analysis из AnalyzerNode
        Output: state с answer (текстовый ответ)
        """

        question = state.get("question", "")
        analysis = state.get("analysis")
        analysis_type = state.get("analysis_type", "general")

        print(f"\n✍️ AnswerGeneratorNode")
        print(f"   Question: {question}")

        if not analysis or not state.get("analysis_success"):
            print("  ❌ Нет анализа для генерации ответа")
            state["answer"] = "К сожалению, не удалось выполнить анализ."
            state["answer_success"] = False
            return state

        try:
            # === Генерируем ответ ===
            if analysis_type == "trend":
                answer = self._generate_trend_answer(question, analysis)
            elif analysis_type == "ranking":
                answer = self._generate_ranking_answer(question, analysis)
            elif analysis_type == "comparison":
                answer = self._generate_comparison_answer(question, analysis)
            elif analysis_type == "aggregation":
                answer = self._generate_aggregation_answer(question, analysis)
            else:
                answer = self._generate_generic_answer(question, analysis)

            state["answer"] = answer
            state["answer_success"] = True

            # === Выводим ответ ===
            self._print_answer(answer)

        except Exception as e:
            print(f"  ❌ Ошибка при генерации ответа: {str(e)}")
            import traceback
            traceback.print_exc()

            state["answer"] = "Произошла ошибка при обработке результатов."
            state["answer_success"] = False
            state["answer_error"] = str(e)

        return state

    def _generate_trend_answer(self, question: str, analysis: Dict[str, Any]) -> str:
        """Генерирует ответ для вопроса про тренд"""

        trend = analysis.get("trend", {})
        insights = analysis.get("insights", [])

        answer_parts = []

        # === Основной вывод ===
        direction = "🔴 снизилась" if trend.get("percent_change", 0) < 0 else "🟢 возросла"
        answer_parts.append(
            f"Цена авиабилетов в 2024 году {direction} на {abs(trend.get('percent_change', 0)):.1f}%"
        )

        # === Детали ===
        answer_parts.append(
            f"\n📊 Детали анализа:"
        )
        answer_parts.append(
            f"• начало года (1 января): {trend.get('start_value', 0):.0f} руб."
        )
        answer_parts.append(
            f"• Конец периода ({trend.get('end_date', 'N/A')}): {trend.get('end_value', 0):.0f} руб."
        )
        answer_parts.append(
            f"• Максимальная цена: {trend.get('max_value', 0):.0f} руб. ({trend.get('max_date', 'N/A')})"
        )
        answer_parts.append(
            f"• Минимальная цена: {trend.get('min_value', 0):.0f} руб. ({trend.get('min_date', 'N/A')})"
        )
        answer_parts.append(
            f"• Средняя цена: {analysis.get('statistics', {}).get(trend.get('metric_column'), {}).get('mean', 0):.0f} руб."
        )

        # === Предупреждения ===
        for insight in insights:
            if "Высокая волатильность" in insight or "⚠️" in insight:
                answer_parts.append(f"\n⚠️ Важно: Цена характеризуется высокой волатильностью, что указывает на нестабильность рынка авиабилетов.")
                break

        # === Общий вывод ===
        if trend.get("percent_change", 0) > 50:
            answer_parts.append(
                f"\n💡 Вывод: Значительный рост цен на авиабилеты. Это может быть связано с сезонностью, ростом спроса или изменением себестоимости."
            )
        elif trend.get("percent_change", 0) < -20:
            answer_parts.append(
                f"\n💡 Вывод: Существенное снижение цен. Это благоприятно для потребителей, ищущих доступные авиабилеты."
            )
        else:
            answer_parts.append(
                f"\n💡 Вывод: Цены относительно стабильны с умеренным изменением в течение года."
            )

        return "\n".join(answer_parts)

    def _generate_ranking_answer(self, question: str, analysis: Dict[str, Any]) -> str:
        """Генерирует ответ для вопроса про ранжирование"""

        ranking = analysis.get("ranking", {})

        answer_parts = [
            f"🏆 Лидер: {ranking.get('top_item', 'N/A')}",
            f"\n📊 Результаты:",
            f"• Значение: {ranking.get('top_value', 0):.2f}",
            f"• Доля от общего: {ranking.get('top_percentage', 0):.1f}%",
            f"• Всего элементов: {ranking.get('total_items', 0)}"
        ]

        return "\n".join(answer_parts)

    def _generate_comparison_answer(self, question: str, analysis: Dict[str, Any]) -> str:
        """Генерирует ответ для вопроса про сравнение"""

        comparison = analysis.get("comparison", {})

        answer_parts = [
            f"⚖️ Сравнительный анализ:",
            f"\n• Минимальное значение: {comparison.get('min_value', 0):.2f}",
            f"• Максимальное значение: {comparison.get('max_value', 0):.2f}",
            f"• Разброс: {comparison.get('spread', 0):.2f} ({comparison.get('spread_percent', 0):.1f}%)",
            f"• Количество элементов: {comparison.get('items_count', 0)}"
        ]

        return "\n".join(answer_parts)

    def _generate_aggregation_answer(self, question: str, analysis: Dict[str, Any]) -> str:
        """Генерирует ответ для вопроса про агрегацию"""

        question_lower = question.lower()
        is_discount = any(word in question_lower for word in ["скидк", "промокод", "discount", "promo"])

        if is_discount:
            # ← СПЕЦИАЛЬНЫЙ ОТВЕТ ДЛЯ СКИДОК
            aggregation = analysis.get("aggregation", {})
            mean_value = aggregation.get("value", 0)
            count_value = aggregation.get("count_rows", 0)

            answer_parts = [
                "💰 Анализ скидок по промокодам",
                f"\n📊 Результаты:",
                f"• Средняя скидка (для заказов со скидкой): {mean_value:.2f} руб.",
            ]

            if count_value:
                answer_parts.append(f"• Количество заказов со скидкой: {count_value}")

            answer_parts.extend([
                f"\n💡 Интерпретация:",
                f"Средняя скидка по промокодам составляет {mean_value:.2f} рублей для заказов, имеющих активный промокод."
            ])

            return "\n".join(answer_parts)

        else:
            # Общий ответ
            aggregation = analysis.get("aggregation", {})
            answer_parts = [
                f"∑ Итоговые данные:",
                f"\n• {aggregation.get('metric_column', 'Значение')}: {aggregation.get('value', 0):.2f}",
                f"• Количество записей: {aggregation.get('total_rows', 0)}"
            ]
            return "\n".join(answer_parts)

    def _generate_generic_answer(self, question: str, analysis: Dict[str, Any]) -> str:
        """Генерирует общий ответ"""

        insights = analysis.get("insights", [])

        answer_parts = [f"📊 Результаты анализа:\n"]

        for insight in insights:
            answer_parts.append(f"• {insight}")

        return "\n".join(answer_parts)

    def _print_answer(self, answer: str):
        """Выводит ответ красиво"""

        print(f"\n  ✓ Ответ сгенерирован")
        print(f"\n" + "=" * 80)
        print(f"📝 ОТВЕТ НА ВОПРОС")
        print(f"=" * 80)
        print(f"\n{answer}\n")
        print(f"=" * 80)

# State

In [42]:
"""
LangGraph Workflow - исправленная версия с правильной работой State
"""
from typing import Dict, Any, Optional, Literal
from dataclasses import dataclass, field, asdict
from datetime import datetime
import json
from langgraph.graph import StateGraph, END
from dataclasses import asdict

# ============= STATE DEFINITION =============
@dataclass
class AnalysisState:
    """
    Состояние всего процесса анализа.
    Используется в LangGraph StateGraph.
    """
    # === INPUT ===
    question: str = ""
    user_id: str = ""
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    # === INTERPRETER OUTPUT ===
    intentions: Dict[str, Any] = field(default_factory=dict)
    schema_context: str = ""
    retrieved_chunks: list = field(default_factory=list)

    # === CLARIFICATION ===
    clarification_needed: bool = False
    ambiguities: list = field(default_factory=list)
    clarification_question: Optional[str] = None
    clarifications_applied: bool = False
    clarifications: Dict[str, Any] = field(default_factory=dict)

    # === PLANNER OUTPUT ===
    plan: Optional[Dict[str, Any]] = None
    plan_valid: bool = False
    analysis_type: str = ""

    # === QUERY BUILDER OUTPUT ===
    query: Optional[Dict[str, Any]] = None
    query_valid: bool = False

    # === EXECUTOR OUTPUT ===
    execution_result: Optional[Dict[str, Any]] = None
    execution_success: bool = False
    execution_error: Optional[str] = None

    # === ANALYZER OUTPUT ===
    analysis: Optional[Dict[str, Any]] = None
    analysis_success: bool = False
    analysis_error: Optional[str] = None

    # === ANSWER GENERATOR OUTPUT ===
    answer: str = ""
    answer_success: bool = False
    answer_error: Optional[str] = None

    # === METADATA ===
    selected_currency: Optional[str] = None
    selected_date_column: Optional[str] = None
    price_column: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        """Конвертирует state в словарь"""
        return asdict(self)


from dataclasses import asdict

# ============= ROUTING FUNCTIONS (ИСПРАВЛЕННЫЕ) =============

def should_ask_clarification(state) -> Literal["clarification", "planner"]:
    """
    Решает, нужны ли уточнения после Interpreter.
    """
    # Конвертируем state в dict если это AnalysisState
    if isinstance(state, dict):
        state_dict = state
    else:
        state_dict = asdict(state)

    clarification_needed = state_dict.get("clarification_needed", False)

    if clarification_needed:
        print("\n🔀 ROUTING: Обнаружены неоднозначности → CLARIFICATION")
        return "clarification"
    else:
        print("\n🔀 ROUTING: Неоднозначностей нет → PLANNER")
        return "planner"


def should_continue_after_planner(state) -> Literal["query_builder", "end"]:
    """
    Решает, продолжать ли после Planner.
    """
    if isinstance(state, dict):
        state_dict = state
    else:
        state_dict = asdict(state)

    plan_valid = state_dict.get("plan_valid", False)

    if plan_valid:
        print("\n🔀 ROUTING: План валиден → QUERY_BUILDER")
        return "query_builder"
    else:
        print("\n🔀 ROUTING: План невалиден → END")
        return "end"


def should_continue_after_query_builder(state) -> Literal["executor", "end"]:
    """
    Решает, продолжать ли после Query Builder.
    """
    if isinstance(state, dict):
        state_dict = state
    else:
        state_dict = asdict(state)

    query_valid = state_dict.get("query_valid", False)

    if query_valid:
        print("\n🔀 ROUTING: Query валиден → EXECUTOR")
        return "executor"
    else:
        print("\n🔀 ROUTING: Query невалиден → END")
        return "end"


def should_continue_after_executor(state) -> Literal["analyzer", "end"]:
    """
    Решает, продолжать ли после Executor.
    """
    if isinstance(state, dict):
        state_dict = state
    else:
        state_dict = asdict(state)

    execution_success = state_dict.get("execution_success", False)

    if execution_success:
        print("\n🔀 ROUTING: Выполнение успешно → ANALYZER")
        return "analyzer"
    else:
        print("\n🔀 ROUTING: Ошибка выполнения → END")
        return "end"


def should_continue_after_analyzer(state) -> Literal["answer_generator", "end"]:
    """
    Решает, продолжать ли после Analyzer.
    """
    if isinstance(state, dict):
        state_dict = state
    else:
        state_dict = asdict(state)

    analysis_success = state_dict.get("analysis_success", False)

    if analysis_success:
        print("\n🔀 ROUTING: Анализ успешен → ANSWER_GENERATOR")
        return "answer_generator"
    else:
        print("\n🔀 ROUTING: Ошибка анализа → END")
        return "end"

# Graph

In [59]:
# ============= GRAPH BUILDER =============

class AnalysisWorkflow:
    """
    Основной workflow граф анализа с использованием LangGraph.
    """

    def __init__(
        self,
        interpreter_node_obj,
        clarification_handler,
        planner_node_obj,
        query_builder_node_obj,
        executor_node_obj,
        analyzer_node_obj,
        answer_generator_node_obj
    ):
        """
        Args:
            interpreter_node_obj: InterpreterNode
            clarification_handler: ClarificationHandler
            planner_node_obj: PlannerNode
            query_builder_node_obj: QueryBuilderNode
            executor_node_obj: ExecutorNode
            analyzer_node_obj: AnalyzerNode
            answer_generator_node_obj: AnswerGeneratorNode
        """
        self.interpreter_obj = interpreter_node_obj
        self.clarification_handler_obj = clarification_handler
        self.planner_obj = planner_node_obj
        self.query_builder_obj = query_builder_node_obj
        self.executor_obj = executor_node_obj
        self.analyzer_obj = analyzer_node_obj
        self.answer_generator_obj = answer_generator_node_obj

        # Строим граф
        self.graph = self._build_graph()

    def _build_graph(self):
        """
        Строит LangGraph StateGraph со всеми узлами и рёбрами.
        """
        print("🔧 Строю LangGraph workflow...")

        # === Создаём StateGraph ===
        workflow = StateGraph(AnalysisState)

        # === ДОБАВЛЯЕМ УЗЛЫ (add_node) ===
        print("  📍 Добавляю узлы...")

        workflow.add_node(
            "interpreter",
            self._make_interpreter_wrapper()
        )

        workflow.add_node(
            "clarification",
            self._make_clarification_wrapper()
        )

        workflow.add_node(
            "planner",
            self._make_planner_wrapper()
        )

        workflow.add_node(
            "query_builder",
            self._make_query_builder_wrapper()
        )

        workflow.add_node(
            "executor",
            self._make_executor_wrapper()
        )

        workflow.add_node(
            "analyzer",
            self._make_analyzer_wrapper()
        )

        workflow.add_node(
            "answer_generator",
            self._make_answer_generator_wrapper()
        )

        # === ДОБАВЛЯЕМ РЁБРА (add_edge) ===
        print("  🔗 Добавляю рёбра...")

        # Стартовая точка → Interpreter
        workflow.set_entry_point("interpreter")

        # Interpreter → Clarification или Planner (условное маршрутирование)
        workflow.add_conditional_edges(
            "interpreter",
            should_ask_clarification,
            {
                "clarification": "clarification",
                "planner": "planner"
            }
        )

        # Clarification → Planner (после уточнений)
        workflow.add_edge("clarification", "planner")

        # Planner → Query Builder или End
        workflow.add_conditional_edges(
            "planner",
            should_continue_after_planner,
            {
                "query_builder": "query_builder",
                "end": END
            }
        )

        # Query Builder → Executor или End
        workflow.add_conditional_edges(
            "query_builder",
            should_continue_after_query_builder,
            {
                "executor": "executor",
                "end": END
            }
        )

        # Executor → Analyzer или End
        workflow.add_conditional_edges(
            "executor",
            should_continue_after_executor,
            {
                "analyzer": "analyzer",
                "end": END
            }
        )

        # Analyzer → Answer Generator или End
        workflow.add_conditional_edges(
            "analyzer",
            should_continue_after_analyzer,
            {
                "answer_generator": "answer_generator",
                "end": END
            }
        )

        # Answer Generator → End
        workflow.add_edge("answer_generator", END)

        # === КОМПИЛИРУЕМ ГРАФ ===
        graph = workflow.compile()

        print("  ✓ Граф построен успешно")
        return graph

    def _make_interpreter_wrapper(self):
        """Создаёт wrapper для Interpreter узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            # Конвертируем state в dict
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state

            # Выполняем узел
            result_dict = self.interpreter_obj.execute(state_dict)

            # Конвертируем обратно в AnalysisState
            return AnalysisState(**result_dict)
        return node_func

    def _make_clarification_wrapper(self):
        """Создаёт wrapper для Clarification узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state

            if not state_dict.get("clarification_needed"):
                return state

            ambiguities = state_dict.get("ambiguities", [])
            clarifications = self.clarification_handler_obj.collect_all_clarifications(ambiguities)
            result_dict = self.clarification_handler_obj.apply_clarifications_to_state(state_dict, clarifications)

            return AnalysisState(**result_dict)
        return node_func

    def _make_planner_wrapper(self):
        """Создаёт wrapper для Planner узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state
            result_dict = self.planner_obj.execute(state_dict)
            return AnalysisState(**result_dict)
        return node_func

    def _make_query_builder_wrapper(self):
        """Создаёт wrapper для Query Builder узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state
            result_dict = self.query_builder_obj.execute(state_dict)
            return AnalysisState(**result_dict)
        return node_func

    def _make_executor_wrapper(self):
        """Создаёт wrapper для Executor узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state
            result_dict = self.executor_obj.execute(state_dict)
            return AnalysisState(**result_dict)
        return node_func

    def _make_analyzer_wrapper(self):
        """Создаёт wrapper для Analyzer узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state
            result_dict = self.analyzer_obj.execute(state_dict)
            return AnalysisState(**result_dict)
        return node_func

    def _make_answer_generator_wrapper(self):
        """Создаёт wrapper для Answer Generator узла"""
        def node_func(state: AnalysisState) -> AnalysisState:
            state_dict = asdict(state) if isinstance(state, AnalysisState) else state
            result_dict = self.answer_generator_obj.execute(state_dict)
            return AnalysisState(**result_dict)
        return node_func

    def invoke(self, question: str, user_id: str = "default") -> Dict[str, Any]:
        """
        Выполняет workflow.
        """
        print("\n" + "=" * 80)
        print("🚀 НАЧИНАЮ ВЫПОЛНЕНИЕ WORKFLOW")
        print("=" * 80)

        initial_state = AnalysisState(
            question=question,
            user_id=user_id,
            timestamp=datetime.now().isoformat()
        )

        try:
            final_state = self.graph.invoke(initial_state)

            print("\n" + "=" * 80)
            print("✅ WORKFLOW ВЫПОЛНЕН УСПЕШНО")
            print("=" * 80)

            # LangGraph возвращает dict, просто вернём его
            return final_state

        except Exception as e:
            print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА: {str(e)}")
            import traceback
            traceback.print_exc()

            return {
                "question": question,
                "user_id": user_id,
                "answer": f"Критическая ошибка при выполнении анализа: {str(e)}",
                "answer_success": False
            }


# ============= ИНИЦИАЛИЗАЦИЯ =============

def init_analysis_workflow(df, schema_dict, rag, llm, cache):
    """
    Инициализирует полный workflow со всеми узлами и рёбрами.
    """
    print("🔧 Инициализирую анализ workflow...")

    # Инициализируем все узлы
    interpreter_node_obj = InterpreterNode(llm, rag, schema_dict, df)
    clarification_handler = ClarificationHandler()
    planner_node_obj = PlannerNode(llm)
    query_builder_node_obj = QueryBuilderNode(df)
    executor_node_obj = init_executor_node(df, schema_dict, cache)
    analyzer_node_obj = AnalyzerNode()
    answer_generator_node_obj = AnswerGeneratorNode(llm)

    # Создаём workflow
    workflow = AnalysisWorkflow(
        interpreter_node_obj=interpreter_node_obj,
        clarification_handler=clarification_handler,
        planner_node_obj=planner_node_obj,
        query_builder_node_obj=query_builder_node_obj,
        executor_node_obj=executor_node_obj,
        analyzer_node_obj=analyzer_node_obj,
        answer_generator_node_obj=answer_generator_node_obj
    )

    print("✓ Workflow инициализирован успешно")
    return workflow

In [122]:
cache.clear()

In [ ]:
# Как менялась цена авиабилетов в 2024 году?
# Какое среднее значение скидки по промокоду?

In [45]:
def example_usage():
    """
    Пример использования полного workflow.
    """
    # Создаём workflow
    workflow = init_analysis_workflow(df, schema_dict, rag, llm, cache)

    # Выполняем анализ
    question = "Максимальный размер скидки в 2024 году"
    result = workflow.invoke(question, user_id="user123")

    # Выводим результат
    print("\n" + "=" * 80)
    print("📊 ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
    print("=" * 80)
    print(f"Вопрос: {result['question']}")
    print(f"Тип анализа: {result['analysis_type']}")
    print(f"Статус: {'✅ Успешно' if result['answer_success'] else '❌ Ошибка'}")
    print(f"\n{result['answer']}")
    print("=" * 80)

# Теперь запустите:
example_usage()

🔧 Инициализирую анализ workflow...
🔧 Строю LangGraph workflow...
  📍 Добавляю узлы...
  🔗 Добавляю рёбра...
  ✓ Граф построен успешно
✓ Workflow инициализирован успешно

🚀 НАЧИНАЮ ВЫПОЛНЕНИЕ WORKFLOW

🔍 InterpreterNode: Максимальный размер скидки в 2024 году
  📚 Ищу релевантные описания колонок...
  ✓ Найдено 5 релевантных описаний
  🤖 Анализирую вопрос с LLM...
    ✓ Найден валидный JSON на позиции 30
  ✓ JSON успешно распарсен
  ✓ Добавлена колонка: promo_code_discount_amt
  🔎 Проверяю неоднозначности...

🔀 ROUTING: Неоднозначностей нет → PLANNER

📋 PlannerNode
   Question: Максимальный размер скидки в 2024 году
  📊 Анализ типа вопроса:
     - Тренд: False
     - Ранжирование: False
     - Агрегация: False
     - Сравнение: False
    → Составляю план АГРЕГАЦИИ
  ✓ План составлен с 1 шагами

🔀 ROUTING: План валиден → QUERY_BUILDER

🔨 QueryBuilderNode
   Question: Максимальный размер скидки в 2024 году
  📝 Обрабатываю 1 шагов плана...
    ✓ Агрегация: group_by='None', метрики=1 шт
    

Это тест, который я использовала до создания графа для проверки узлов

In [56]:
def test_planner():
    """
    Полный тест: Interpreter → Planner → QueryBuilder → Executor → Analyzer → AnswerGenerator
    """

    print("\n" + "=" * 80)
    print("🧪 ПОЛНЫЙ ТЕСТ: INTERPRETER → PLANNER → QUERY BUILDER → EXECUTOR → ANALYZER → ANSWER GENERATOR")
    print("=" * 80)

    question = "Какая программа лояльности наиболее активно покупает авиабилеты?"

    # === STEP 1: Interpreter ===
    print("\n1️⃣ INTERPRETER NODE")
    print("-" * 80)

    interpreter = InterpreterNode(llm, rag, schema_dict, df)
    state = {"question": question}
    state = interpreter.execute(state)

    # === STEP 2: Уточнения  ===
    if state["clarification_needed"]:
        print("\n2️⃣ СБОР УТОЧНЕНИЙ")
        print("-" * 80)

        handler = ClarificationHandler()
        clarifications = handler.collect_all_clarifications(state["ambiguities"])

        # === STEP 3: Применяем уточнения ===
        print("\n3️⃣ ПРИМЕНЕНИЕ УТОЧНЕНИЙ")
        print("-" * 80)

        state = handler.apply_clarifications_to_state(state, clarifications)

    # === STEP 4: Planner ===
    print("\n4️⃣ PLANNER NODE")
    print("-" * 80)

    planner = PlannerNode(llm)
    state = planner.execute(state)

    if not state["plan_valid"]:
        print("\n❌ План не был составлен")
        return state

    # === STEP 5: QueryBuilder ===
    print("\n5️⃣ QUERY BUILDER NODE")
    print("-" * 80)

    query_builder = QueryBuilderNode(df)
    state = query_builder.execute(state)

    if not state["query_valid"]:
        print("\n❌ Query невалиден")
        return state

    # === STEP 6: Executor ===
    print("\n6️⃣ EXECUTOR NODE")
    print("-" * 80)

    executor_node = ExecutorNode(SimpleAnalyticsExecutor(df, schema_dict, cache))
    state = executor_node.execute(state)

    if not state["execution_success"]:
        print("\n❌ Выполнение завершилось ошибкой")
        return state

    # === STEP 7: Analyzer ===
    print("\n7️⃣ ANALYZER NODE")
    print("-" * 80)

    analyzer = AnalyzerNode()
    state = analyzer.execute(state)

    if not state["analysis_success"]:
        print("\n❌ Анализ завершился ошибкой")
        return state

    # === STEP 8: AnswerGenerator ===
    print("\n8️⃣ ANSWER GENERATOR NODE")
    print("-" * 80)

    answer_generator = AnswerGeneratorNode()
    state = answer_generator.execute(state)

    # === Финальная информация ===
    if state.get("answer_success"):
        print("\n" + "=" * 80)
        print("✅ ПОЛНЫЙ PIPELINE ЗАВЕРШЁН УСПЕШНО")
        print("=" * 80)
    else:
        print("\n❌ Генерация ответа завершилась ошибкой")

    return state


if __name__ == "__main__":
    state = test_planner()


🧪 ПОЛНЫЙ ТЕСТ: INTERPRETER → PLANNER → QUERY BUILDER → EXECUTOR → ANALYZER → ANSWER GENERATOR

1️⃣ INTERPRETER NODE
--------------------------------------------------------------------------------

🔍 InterpreterNode: Какая программа лояльности наиболее активно покупает авиабилеты?
  📚 Ищу релевантные описания колонок...
  ✓ Найдено 5 релевантных описаний
  🤖 Анализирую вопрос с LLM...
    ✓ Найден валидный JSON на позиции 30
  ✓ JSON успешно распарсен
  ✓ Добавлен автофильтр: order_type_cd == 'AIR'
  🔎 Проверяю неоднозначности...

4️⃣ PLANNER NODE
--------------------------------------------------------------------------------

📋 PlannerNode
   Question: Какая программа лояльности наиболее активно покупает авиабилеты?
  📊 Анализ типа вопроса:
     - Тренд: False
     - Ранжирование: True
     - Агрегация: False
     - Сравнение: False
    → Составляю план РАНЖИРОВАНИЯ
  ✓ План составлен с 4 шагами

5️⃣ QUERY BUILDER NODE
----------------------------------------------------------------

# Сложные вопросы

In [123]:
# -*- coding: utf-8 -*-
"""НОВЫЙ КОД: Агент-Исследователь для сложных вопросов
Добавьте эту ячейку в конец ноутбука и запустите
"""

import json
import re
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from datetime import datetime
import time

# ============================================================================
# НОВЫЙ SUPERVISOR NODE - определяет тип вопроса
# ============================================================================

class SupervisorNode:
    """
    Определяет тип вопроса: простой аналитический или сложное исследование
    """

    def __init__(self, llm):
        self.llm = llm

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        question = state.get("question", "")

        print(f"\n🤖 SupervisorNode: Определяю тип вопроса...")
        print(f"   Вопрос: {question}")

        # Сначала проверяем по ключевым словам (быстрая эвристика)
        question_lower = question.lower()

        # Ключевые слова для простых аналитических вопросов
        simple_keywords = [
            "сколько", "какой", "какая", "какое", "какие",
            "средний", "средняя", "среднее", "максимальный", "минимальный",
            "цена", "стоимость", "скидка", "количество",
            "измен", "менял", "динамик", "тренд",
            "программа лояльности", "loyalty"
        ]

        # Ключевые слова для сложных исследовательских вопросов
        complex_keywords = [
            "серых реселлеров", "мошенник", "подозрительный",
            "дарим деньги", "недооценен", "аномалия",
            "портрет", "типичный", "характеристика",
            "критерий", "правило", "закономерность"
        ]

        # Проверяем по ключевым словам
        is_simple = any(keyword in question_lower for keyword in simple_keywords)
        is_complex = any(keyword in question_lower for keyword in complex_keywords)

        # Если есть явные признаки сложного вопроса
        if is_complex and not is_simple:
            question_type = "complex_investigation"
            print(f"   → По ключевым словам: complex_investigation")
        # Если есть явные признаки простого вопроса
        elif is_simple and not is_complex:
            question_type = "simple_analytics"
            print(f"   → По ключевым словам: simple_analytics")
        else:
            # Если неопределенно, используем LLM
            prompt = f"""Ты - маршрутизатор запросов. Определи тип вопроса пользователя.

ВОПРОС: {question}

ПРАВИЛА КЛАССИФИКАЦИИ:
- "simple_analytics" - если вопрос про:
  * конкретные метрики (среднее, максимум, минимум, сумма)
  * изменение цен/показателей во времени
  * сравнение конкретных групп
  * простые агрегации
  * программы лояльности, рейтинги, топы
  * Примеры: "Как менялась цена?", "Сколько продаж?", "Какая средняя скидка?", "Какая программа лояльности самая активная?"

- "complex_investigation" - если вопрос требует:
  * определения сложных категорий ("серые реселлеры", "мошенники", "подозрительные")
  * выявления аномалий ("где мы дарим деньги")
  * многофакторного анализа для поиска закономерностей
  * построения профиля или портрета клиента
  * Примеры: "Каких клиентов считать серыми реселлерами?", "Где мы дарим деньги?"

Ответь ТОЛЬКО одним словом: simple_analytics или complex_investigation
"""
            response = self.llm.generate(prompt).strip().lower()
            response = re.sub(r'[^\w_]', '', response)

            if "complex_investigation" in response:
                question_type = "complex_investigation"
            else:
                question_type = "simple_analytics"

            print(f"   → По LLM: {question_type}")

        state["question_type"] = question_type
        return state


# ============================================================================
# ГИПОТЕЗЫ ДЛЯ СЕРЫХ РЕСЕЛЛЕРОВ
# ============================================================================

class GrayResellerHypothesisGenerator:
    """
    Генерирует гипотезы для определения серых реселлеров
    """

    def __init__(self, llm, df, schema_dict):
        self.llm = llm
        self.df = df
        self.schema_dict = schema_dict

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n🕵️ GrayResellerHypothesisGenerator: Генерирую гипотезы...")

        # Статистика по датасету для гипотез
        stats = self._calculate_basic_stats()
        high_freq_threshold = int(stats.get("max_orders_per_client", 50) * 0.8) or 20

        print(f"\n📊 Найденные колонки из PDF:")
        print(f"   ID клиента: client_rk")
        print(f"   ID заказа: order_rk")
        print(f"   Тип заказа: order_type_cd")
        print(f"   Статус заказа: order_status_cd")
        print(f"   Промокод скидка: promo_code_discount_amt")
        print(f"   Город отеля: hotel_city")
        print(f"   Город вылета: avia_dep_city")
        print(f"   Город прилета: avia_arr_city")
        print(f"\n📊 Порог частоты для гипотезы: > {high_freq_threshold} заказов")

        # Формируем prompt с реальными названиями колонок из PDF
        prompt = f"""Ты - эксперт по выявлению мошеннических схем в авиабилетах и отелях.

    Нужно определить критерии для "серых реселлеров" - клиентов, которые покупают билеты/отели для перепродажи, а не для личного пользования.

    Доступные колонки в данных (из PDF-описания):
    - client_rk (идентификатор клиента)
    - order_rk (идентификатор заказа)
    - account_rk (идентификатор счета)
    - order_type_cd (тип заказа: Авиа/Отели)
    - order_status_cd (статус заказа)
    - promo_code_discount_amt (размер скидки по промокоду)
    - created_dttm (дата создания заказа)
    - book_start_dttm (дата заезда для отелей)
    - book_end_dttm (дата выезда для отелей)
    - cancel_dttm (дата отмены заказа)
    - hotel_city (город отеля)
    - hotel_country (страна отеля)
    - avia_dep_city (город вылета)
    - avia_arr_city (город прилета)

    Статистика по данным:
    - Всего записей: {stats['total_rows']}
    - Уникальных клиентов (client_rk): {stats['unique_clients']}
    - Средняя частота покупок на клиента: {stats['avg_orders_per_client']:.2f}
    - Максимальная частота: {stats['max_orders_per_client']}
    - Рекомендуемый порог для "высокой частоты": > {high_freq_threshold} заказов

    Сгенерируй 5-7 гипотез (критериев) для выявления серых реселлеров.
    Каждая гипотеза должна проверять определенный паттерн поведения.

    ВАЖНО: В query_template используй ТОЛЬКО эти названия колонок:
    - client_rk (для группировки по клиентам)
    - order_rk (для подсчета заказов)
    - order_type_cd (для фильтрации по типу)
    - order_status_cd (для проверки статуса)
    - promo_code_discount_amt (для проверки скидок)
    - created_dttm (для временных паттернов)
    - hotel_city, avia_dep_city, avia_arr_city (для проверки направлений)

    Формат ответа - валидный JSON. Вот пример структуры:

    {{
        "hypotheses": [
            {{
                "id": "hyp_1",
                "name": "Высокая частота покупок",
                "description": "Клиент покупает аномально часто",
                "reasoning": "Обычные пассажиры покупают билеты редко, высокая частота указывает на коммерческую деятельность",
                "query_template": "GROUP BY client_rk HAVING COUNT(order_rk) > ПОРОГ_ЧАСТОТЫ",
                "columns_needed": ["client_rk", "order_rk"],
                "priority": 1
            }},
            {{
                "id": "hyp_2",
                "name": "Активное использование промокодов",
                "description": "Клиент использует промокоды в большинстве заказов",
                "reasoning": "Перепродавцы максимизируют маржу, используя скидки",
                "query_template": "SELECT client_rk, AVG(CASE WHEN promo_code_discount_amt > 0 THEN 1.0 ELSE 0.0 END) as promo_rate FROM data GROUP BY client_rk HAVING promo_rate > 0.7",
                "columns_needed": ["client_rk", "order_rk", "promo_code_discount_amt"],
                "priority": 2
            }}
        ]
    }}

    Заполни ПОРОГ_ЧАСТОТЫ конкретным числом {high_freq_threshold} в своей гипотезе.

    Гипотезы должны быть разнообразными: частота, использование промокодов, отмены, разнообразие направлений, временные паттерны и т.д.
    """

        response = self.llm.generate(prompt)

        # Извлекаем JSON
        hypotheses = self._extract_hypotheses(response)

        # Добавляем несколько базовых гипотез на случай, если LLM не сработает
        fallback_hypotheses = self._get_fallback_hypotheses(stats)

        if hypotheses and hypotheses.get("hypotheses"):
            # Объединяем с fallback, убирая дубликаты
            all_hypotheses = hypotheses["hypotheses"] + fallback_hypotheses["hypotheses"]
            # Убираем дубликаты по id
            unique_hyp = {h["id"]: h for h in all_hypotheses}.values()
            state["hypotheses"] = list(unique_hyp)
        else:
            state["hypotheses"] = fallback_hypotheses["hypotheses"]

        print(f"   ✓ Сгенерировано {len(state['hypotheses'])} гипотез")
        for i, h in enumerate(state["hypotheses"][:3], 1):
            print(f"     {i}. {h['name']}")

        state["investigation_type"] = "gray_reseller"
        state["current_hypothesis_index"] = 0
        state["hypothesis_results"] = {}
        state["investigation_complete"] = False

        return state

    def _calculate_basic_stats(self) -> Dict[str, Any]:
        """Рассчитывает базовую статистику по датасету"""
        stats = {}

        try:
            stats["total_rows"] = len(self.df)

            # Используем client_rk из PDF
            if "client_rk" in self.df.columns:
                client_counts = self.df["client_rk"].value_counts()
                stats["unique_clients"] = len(client_counts)
                stats["avg_orders_per_client"] = client_counts.mean()
                stats["max_orders_per_client"] = client_counts.max()
            else:
                stats["unique_clients"] = 0
                stats["avg_orders_per_client"] = 0
                stats["max_orders_per_client"] = 0

        except Exception as e:
            print(f"   ⚠️ Ошибка при расчете статистики: {e}")
            stats = {
                "total_rows": len(self.df),
                "unique_clients": 0,
                "avg_orders_per_client": 0,
                "max_orders_per_client": 0
            }

        return stats

    def _extract_hypotheses(self, response: str) -> Optional[Dict]:
        """Извлекает гипотезы из ответа LLM"""
        json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
        matches = re.finditer(json_pattern, response, re.DOTALL)

        for match in matches:
            try:
                obj = json.loads(match.group(0))
                if "hypotheses" in obj and isinstance(obj["hypotheses"], list):
                    return obj
            except:
                continue
        return None

    def _get_fallback_hypotheses(self, stats) -> Dict:
        """Возвращает базовые гипотезы на случай ошибки LLM"""
        high_freq_threshold = int(stats.get("max_orders_per_client", 50) * 0.8) or 20

        return {
            "hypotheses": [
                {
                    "id": "hyp_1",
                    "name": "Высокая частота покупок",
                    "description": f"Клиент покупает аномально часто (больше {high_freq_threshold} заказов)",
                    "reasoning": "Частота покупок значительно превышает среднюю, что указывает на коммерческую деятельность",
                    "query_template": f"GROUP BY client_rk HAVING COUNT(order_rk) > {high_freq_threshold}",
                    "columns_needed": ["client_rk", "order_rk"],
                    "priority": 1
                },
                {
                    "id": "hyp_2",
                    "name": "Активное использование промокодов",
                    "description": "Клиент использует промокоды в большинстве заказов (>70%)",
                    "reasoning": "Перепродавцы максимизируют маржу, используя скидки",
                    "query_template": "SELECT client_rk, AVG(CASE WHEN promo_code_discount_amt > 0 THEN 1.0 ELSE 0.0 END) as promo_rate FROM df GROUP BY client_rk HAVING promo_rate > 0.7",
                    "columns_needed": ["client_rk", "order_rk", "promo_code_discount_amt"],
                    "priority": 2
                },
                {
                    "id": "hyp_3",
                    "name": "Высокая доля отмен",
                    "description": "Клиент часто отменяет заказы (>30% отмен)",
                    "reasoning": "Реселлеры могут бронировать для проверки цен, а затем отменять",
                    "query_template": "SELECT client_rk, AVG(CASE WHEN order_status_cd = 'CANCELLED' THEN 1.0 ELSE 0.0 END) as cancel_rate FROM df GROUP BY client_rk HAVING cancel_rate > 0.3",
                    "columns_needed": ["client_rk", "order_rk", "order_status_cd"],
                    "priority": 3
                },
                {
                    "id": "hyp_4",
                    "name": "Разнообразие направлений",
                    "description": "Клиент покупает билеты во множество разных городов",
                    "reasoning": "Реселлеры обслуживают клиентов из разных регионов",
                    "query_template": "SELECT client_rk, COUNT(DISTINCT CASE WHEN order_type_cd = 'HOT' THEN hotel_city ELSE avia_arr_city END) as city_count FROM df GROUP BY client_rk HAVING city_count > 10",
                    "columns_needed": ["client_rk", "order_type_cd", "hotel_city", "avia_arr_city"],
                    "priority": 4
                },
                {
                    "id": "hyp_5",
                    "name": "Только авиабилеты или только отели",
                    "description": "Клиент специализируется только на одном типе продуктов",
                    "reasoning": "Реселлеры часто фокусируются на одном типе услуг",
                    "query_template": "SELECT client_rk, COUNT(DISTINCT order_type_cd) as product_types FROM df GROUP BY client_rk HAVING product_types = 1",
                    "columns_needed": ["client_rk", "order_type_cd"],
                    "priority": 5
                }
            ]
        }


# ============================================================================
# ГИПОТЕЗЫ ДЛЯ НЕДООЦЕНЕННЫХ НАПРАВЛЕНИЙ
# ============================================================================

class UndervaluedRoutesHypothesisGenerator:
    """
    Генерирует гипотезы для поиска недооцененных направлений/дат
    """

    def __init__(self, llm, df, schema_dict):
        self.llm = llm
        self.df = df
        self.schema_dict = schema_dict

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n💰 UndervaluedRoutesHypothesisGenerator: Ищу недооцененные направления...")

        # Определяем реальные названия колонок
        city_cols = {
            'hotel_city': 'hotel_city' if 'hotel_city' in self.df.columns else None,
            'avia_dep_city': 'avia_dep_city' if 'avia_dep_city' in self.df.columns else None,
            'avia_arr_city': 'avia_arr_city' if 'avia_arr_city' in self.df.columns else None
        }

        date_cols = {
            'created_dttm': 'created_dttm' if 'created_dttm' in self.df.columns else None,
            'book_start_dttm': 'book_start_dttm' if 'book_start_dttm' in self.df.columns else None
        }

        price_col = 'nominal_price_rub_amt' if 'nominal_price_rub_amt' in self.df.columns else None

        print(f"\n📊 Доступные колонки для анализа:")
        print(f"   Города отелей: {city_cols['hotel_city']}")
        print(f"   Города вылета: {city_cols['avia_dep_city']}")
        print(f"   Города прилета: {city_cols['avia_arr_city']}")
        print(f"   Даты: {[v for v in date_cols.values() if v]}")
        print(f"   Цена: {price_col}")

        # Создаем гипотезы на основе доступных колонок
        hypotheses = []

        # Гипотеза 1: По городам отелей
        if city_cols['hotel_city'] and price_col:
            hypotheses.append({
                "id": "hotel_cities",
                "name": "Недооцененные города отелей",
                "description": "Города отелей с высокой загрузкой, но низкой ценой",
                "group_by": city_cols['hotel_city'],
                "price_col": price_col,
                "type": "hotel"
            })

        # Гипотеза 2: По городам прилета (авиа)
        if city_cols['avia_arr_city'] and price_col:
            hypotheses.append({
                "id": "flight_destinations",
                "name": "Недооцененные направления перелетов",
                "description": "Города прилета с высокой загрузкой, но низкой ценой",
                "group_by": city_cols['avia_arr_city'],
                "price_col": price_col,
                "type": "flight"
            })

        # Гипотеза 3: По дням недели (из created_dttm)
        if date_cols['created_dttm'] and price_col:
            hypotheses.append({
                "id": "day_of_week",
                "name": "Дни недели с низкими ценами",
                "description": "Дни недели, когда цена ниже средней при высокой загрузке",
                "group_by": "day_of_week",
                "price_col": price_col,
                "type": "dow",
                "date_col": date_cols['created_dttm']
            })

        # Гипотеза 4: По месяцам
        if date_cols['created_dttm'] and price_col:
            hypotheses.append({
                "id": "month",
                "name": "Месяцы с сезонными провалами цен",
                "description": "Месяцы, когда цена аномально низкая при высокой загрузке",
                "group_by": "month",
                "price_col": price_col,
                "type": "month",
                "date_col": date_cols['created_dttm']
            })

        state["hypotheses"] = hypotheses
        print(f"   ✓ Сгенерировано {len(hypotheses)} гипотез")
        for i, h in enumerate(hypotheses, 1):
            print(f"     {i}. {h['name']}")

        state["investigation_type"] = "undervalued_routes"
        state["current_hypothesis_index"] = 0
        state["hypothesis_results"] = {}
        state["investigation_complete"] = False

        return state


# ============================================================================
# ИСПОЛНИТЕЛЬ ГИПОТЕЗ
# ============================================================================

class HypothesisTester:
    """
    Проверяет гипотезы на данных
    """

    def __init__(self, df, executor):
        self.df = df
        self.executor = executor

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n🧪 HypothesisTester: Проверяю гипотезы...")

        hypotheses = state.get("hypotheses", [])
        current_idx = state.get("current_hypothesis_index", 0)

        if current_idx >= len(hypotheses):
            print(f"   ✓ Все гипотезы проверены")
            state["investigation_complete"] = True
            return state

        hypothesis = hypotheses[current_idx]
        print(f"   Проверяю гипотезу {current_idx + 1}/{len(hypotheses)}: {hypothesis['name']}")

        # Создаем запрос на основе гипотезы
        query = self._build_query_from_hypothesis(hypothesis, state)

        if query:
            # Выполняем запрос
            result = self.executor.execute_simple_query(query)
            state["hypothesis_results"][hypothesis["id"]] = {
                "hypothesis": hypothesis,
                "result": result,
                "data": result.get("data", [])
            }
            print(f"   ✓ Получено {result.get('row_count', 0)} строк результатов")
        else:
            print(f"   ⚠️ Не удалось построить запрос для гипотезы")

        # Переходим к следующей гипотезе
        state["current_hypothesis_index"] = current_idx + 1

        return state

    def _build_query_from_hypothesis(self, hypothesis: Dict, state: Dict) -> Optional[Dict]:
        """Строит query для проверки гипотезы"""

        inv_type = state.get("investigation_type")

        # Проверяем, что нужные колонки существуют в датасете
        columns_needed = hypothesis.get("columns_needed", [])
        available_columns = set(self.df.columns)

        missing_columns = [col for col in columns_needed if col not in available_columns]
        if missing_columns:
            print(f"   ⚠️ Внимание: отсутствуют колонки {missing_columns} для гипотезы {hypothesis.get('name')}")
            # Продолжаем, но с предупреждением

        if inv_type == "gray_reseller":
            return self._build_gray_reseller_query(hypothesis)
        elif inv_type == "undervalued_routes":
            return self._build_undervalued_routes_query(hypothesis)
        else:
            return None

    def _build_gray_reseller_query(self, hypothesis: Dict) -> Dict:
        """Строит запрос для поиска серых реселлеров"""

        query = {
            "filters": [],
            "group_by": "client_rk",  # Используем правильное название из PDF
            "metrics": [
                {
                    "type": "count",
                    "name": "order_count"
                }
            ],
            "sort_by": "order_count",
            "sort_order": "desc",
            "limit": 100
        }

        # Добавляем специфичные метрики в зависимости от гипотезы
        hyp_name = hypothesis.get("name", "").lower()
        columns_needed = hypothesis.get("columns_needed", [])

        if "промокод" in hyp_name or "promo" in hyp_name:
            if "promo_code_discount_amt" in columns_needed:
                query["metrics"].append({
                    "type": "mean",
                    "column": "promo_code_discount_amt",
                    "name": "avg_discount"
                })

        if "отмен" in hyp_name or "cancel" in hyp_name:
            # Для отмен нужно будет сделать отдельный запрос
            pass

        return query

    def _build_undervalued_routes_query(self, hypothesis: Dict) -> Dict:
        """Строит запрос для поиска недооцененных направлений"""

        group_by = hypothesis.get("group_by")
        price_col = hypothesis.get("price_col", "nominal_price_rub_amt")

        # Для разных типов гипотез строим разные запросы
        hyp_type = hypothesis.get("type")

        if hyp_type == "hotel":
            # Для отелей группируем по городу
            query = {
                "filters": [{"column": "order_type_cd", "value": "HOT", "op": "=="}],
                "group_by": group_by,
                "metrics": [
                    {"type": "mean", "column": price_col, "name": "avg_price"},
                    {"type": "count", "name": "booking_count"}
                ],
                "sort_by": "booking_count",
                "sort_order": "desc",
                "limit": 50
            }

        elif hyp_type == "flight":
            # Для авиа группируем по городу прилета
            query = {
                "filters": [{"column": "order_type_cd", "value": "AIR", "op": "=="}],
                "group_by": "avia_arr_city",
                "metrics": [
                    {"type": "mean", "column": price_col, "name": "avg_price"},
                    {"type": "count", "name": "booking_count"}
                ],
                "sort_by": "booking_count",
                "sort_order": "desc",
                "limit": 50
            }

        elif hyp_type == "dow":
            # Для дней недели - нужно извлечь день из даты
            # Используем простой подход - группируем по дате и потом обработаем
            date_col = hypothesis.get("date_col", "created_dttm")
            print(f"   📅 Анализ по дням недели на основе {date_col}...")

            # Временно группируем по дате, потом обработаем отдельно
            query = {
                "filters": [],
                "group_by": date_col,
                "metrics": [
                    {"type": "mean", "column": price_col, "name": "avg_price"},
                    {"type": "count", "name": "booking_count"}
                ],
                "sort_by": date_col,
                "sort_order": "asc",
                "limit": 1000  # Берем больше данных для анализа
            }

        elif hyp_type == "month":
            # Для месяцев - нужно извлечь месяц из даты
            date_col = hypothesis.get("date_col", "created_dttm")
            print(f"   📅 Анализ по месяцам на основе {date_col}...")

            # Временно группируем по дате, потом обработаем отдельно
            query = {
                "filters": [],
                "group_by": date_col,
                "metrics": [
                    {"type": "mean", "column": price_col, "name": "avg_price"},
                    {"type": "count", "name": "booking_count"}
                ],
                "sort_by": date_col,
                "sort_order": "asc",
                "limit": 1000
            }

        else:
            # Общий случай
            query = {
                "filters": [],
                "group_by": group_by,
                "metrics": [
                    {"type": "mean", "column": price_col, "name": "avg_price"},
                    {"type": "count", "name": "booking_count"}
                ],
                "sort_by": "booking_count",
                "sort_order": "desc",
                "limit": 50
            }

        return query


# ============================================================================
# СИНТЕЗАТОР РЕЗУЛЬТАТОВ
# ============================================================================

class EvidenceSynthesizer:
    """
    Синтезирует результаты проверки гипотез в итоговый ответ
    """

    def __init__(self, llm):
        self.llm = llm

    def execute(self, state: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n🔬 EvidenceSynthesizer: Анализирую результаты...")

        inv_type = state.get("investigation_type")
        results = state.get("hypothesis_results", {})

        if not results:
            print("   ❌ Нет результатов для анализа")
            state["analysis"] = {"error": "No results to synthesize"}
            state["analysis_success"] = True  # Меняем на True, чтобы продолжить
            # Генерируем базовый ответ даже без результатов
            state["answer"] = "Недостаточно данных для анализа. Попробуйте уточнить вопрос."
            state["answer_success"] = True
            return state

        if inv_type == "gray_reseller":
            synthesis = self._synthesize_gray_reseller(results, state)
        elif inv_type == "undervalued_routes":
            synthesis = self._synthesize_undervalued_routes(results, state)
        else:
            synthesis = {"error": "Unknown investigation type"}

        state["analysis"] = synthesis
        state["analysis_success"] = True

        # Генерируем текстовый ответ на основе синтеза
        state["answer"] = self._generate_answer(synthesis, state.get("question", ""))
        state["answer_success"] = True

        print(f"\n✅ Синтез завершен. Сгенерирован ответ длиной {len(state['answer'])} символов")

        return state

    def _synthesize_gray_reseller(self, results: Dict, state: Dict) -> Dict:
        """Синтезирует результаты для серых реселлеров"""

        # Собираем всех подозрительных клиентов из разных гипотез
        suspicious_clients = {}
        hypothesis_stats = {}

        for hyp_id, hyp_result in results.items():
            data = hyp_result.get("data", [])
            hyp_name = hyp_result.get("hypothesis", {}).get("name", hyp_id)

            # Сохраняем статистику по гипотезе
            hypothesis_stats[hyp_name] = {
                "total_found": len(data),
                "sample_clients": [row.get("client_rk") for row in data[:5] if "client_rk" in row]
            }

            for row in data:
                client_id = row.get("client_rk")
                if client_id:
                    if client_id not in suspicious_clients:
                        suspicious_clients[client_id] = {
                            "client_rk": client_id,
                            "matched_hypotheses": [],
                            "order_count": row.get("order_count", 0)
                        }
                    suspicious_clients[client_id]["matched_hypotheses"].append(hyp_name)

        # Сортируем по количеству совпадений
        ranked_clients = sorted(
            suspicious_clients.values(),
            key=lambda x: len(x["matched_hypotheses"]),
            reverse=True
        )

        # Берем топ-20 для анализа
        top_clients = ranked_clients[:20]

        # Считаем статистику по уровням риска
        risk_levels = {
            "high_risk": len([c for c in ranked_clients if len(c["matched_hypotheses"]) >= 4]),
            "medium_risk": len([c for c in ranked_clients if len(c["matched_hypotheses"]) == 3]),
            "low_risk": len([c for c in ranked_clients if len(c["matched_hypotheses"]) == 2]),
            "potential_risk": len([c for c in ranked_clients if len(c["matched_hypotheses"]) == 1])
        }

        # Находим клиентов, попавших во все гипотезы
        all_hypotheses_count = len(results)
        most_suspicious = [c for c in ranked_clients if len(c["matched_hypotheses"]) == all_hypotheses_count]

        synthesis = {
            "type": "gray_reseller_analysis",
            "total_clients_analyzed": len(suspicious_clients),
            "risk_levels": risk_levels,
            "most_suspicious_clients": most_suspicious[:5],
            "top_suspicious_clients": top_clients[:10],
            "hypothesis_statistics": hypothesis_stats,
            "methodology": "Клиенты считаются подозрительными, если они попадают в несколько гипотез. Чем больше гипотез подтверждается, тем выше риск."
        }

        # Добавляем портрет типичного подозрительного клиента
        if top_clients:
            synthesis["typical_profile"] = self._generate_reseller_profile(top_clients)

        return synthesis

    def _synthesize_undervalued_routes(self, results: Dict, state: Dict) -> Dict:
        """Синтезирует результаты для недооцененных направлений"""

        undervalued_items = {
            "hotel_cities": [],
            "flight_destinations": [],
            "days_of_week": {},
            "months": {}
        }

        # Собираем сырые данные
        for hyp_id, hyp_result in results.items():
            data = hyp_result.get("data", [])
            if not data:
                continue

            hyp_data = hyp_result.get("hypothesis", {})
            hyp_type = hyp_data.get("type")

            print(f"\n   📊 Обработка гипотезы типа: {hyp_type}")

            if hyp_type == "hotel":
                # Для отелей
                for row in data:
                    if "hotel_city" in row and "avg_price" in row and "booking_count" in row:
                        undervalued_items["hotel_cities"].append({
                            "city": row.get("hotel_city", "unknown"),
                            "avg_price": float(row.get("avg_price", 0)),
                            "booking_count": int(row.get("booking_count", 0))
                        })
                print(f"      Собрано городов отелей: {len(undervalued_items['hotel_cities'])}")

            elif hyp_type == "flight":
                # Для авианаправлений
                for row in data:
                    if "avia_arr_city" in row and "avg_price" in row and "booking_count" in row:
                        undervalued_items["flight_destinations"].append({
                            "destination": row.get("avia_arr_city", "unknown"),
                            "avg_price": float(row.get("avg_price", 0)),
                            "booking_count": int(row.get("booking_count", 0))
                        })
                print(f"      Собрано направлений: {len(undervalued_items['flight_destinations'])}")

            elif hyp_type == "dow":
                # Для дней недели
                print(f"      Обработка дней недели из {len(data)} строк")
                for row in data:
                    # Ищем колонку с датой (первая колонка)
                    date_col = None
                    for col in row.keys():
                        if col not in ["avg_price", "booking_count"]:
                            date_col = col
                            break

                    if date_col and row[date_col]:
                        try:
                            import pandas as pd
                            from datetime import datetime

                            # Пробуем разные форматы дат
                            date_val = row[date_col]
                            if isinstance(date_val, str):
                                date = pd.to_datetime(date_val)
                            else:
                                date = pd.to_datetime(str(date_val))

                            dow = date.dayofweek  # 0=понедельник, 6=воскресенье
                            dow_names = ["Понедельник", "Вторник", "Среда", "Четверг", "Пятница", "Суббота", "Воскресенье"]
                            dow_name = dow_names[dow]

                            if dow_name not in undervalued_items["days_of_week"]:
                                undervalued_items["days_of_week"][dow_name] = {
                                    "total_price": 0,
                                    "total_bookings": 0,
                                    "count": 0
                                }

                            undervalued_items["days_of_week"][dow_name]["total_price"] += float(row.get("avg_price", 0)) * int(row.get("booking_count", 1))
                            undervalued_items["days_of_week"][dow_name]["total_bookings"] += int(row.get("booking_count", 0))
                            undervalued_items["days_of_week"][dow_name]["count"] += 1
                        except Exception as e:
                            continue

                print(f"      Собрано дней недели: {len(undervalued_items['days_of_week'])}")

            elif hyp_type == "month":
                # Для месяцев
                print(f"      Обработка месяцев из {len(data)} строк")
                month_names = ["Январь", "Февраль", "Март", "Апрель", "Май", "Июнь",
                              "Июль", "Август", "Сентябрь", "Октябрь", "Ноябрь", "Декабрь"]

                for row in data:
                    # Ищем колонку с датой
                    date_col = None
                    for col in row.keys():
                        if col not in ["avg_price", "booking_count"]:
                            date_col = col
                            break

                    if date_col and row[date_col]:
                        try:
                            import pandas as pd
                            date = pd.to_datetime(row[date_col])
                            month = date.month - 1  # 0-11
                            month_name = month_names[month]

                            if month_name not in undervalued_items["months"]:
                                undervalued_items["months"][month_name] = {
                                    "total_price": 0,
                                    "total_bookings": 0,
                                    "count": 0
                                }

                            undervalued_items["months"][month_name]["total_price"] += float(row.get("avg_price", 0)) * int(row.get("booking_count", 1))
                            undervalued_items["months"][month_name]["total_bookings"] += int(row.get("booking_count", 0))
                            undervalued_items["months"][month_name]["count"] += 1
                        except Exception as e:
                            continue

                print(f"      Собрано месяцев: {len(undervalued_items['months'])}")

        # Преобразуем словари в списки для анализа
        days_list = []
        for day, stats in undervalued_items["days_of_week"].items():
            if stats["count"] > 0:
                days_list.append({
                    "name": day,
                    "avg_price": stats["total_price"] / stats["count"],
                    "booking_count": stats["total_bookings"]
                })

        months_list = []
        for month, stats in undervalued_items["months"].items():
            if stats["count"] > 0:
                months_list.append({
                    "name": month,
                    "avg_price": stats["total_price"] / stats["count"],
                    "booking_count": stats["total_bookings"]
                })

        print(f"\n   📊 Итоговые данные для анализа:")
        print(f"      Отели: {len(undervalued_items['hotel_cities'])} городов")
        print(f"      Авиа: {len(undervalued_items['flight_destinations'])} направлений")
        print(f"      Дни недели: {len(days_list)}")
        print(f"      Месяцы: {len(months_list)}")

        # Находим недооцененные
        result = {
            "type": "undervalued_routes_analysis",
            "hotel_cities": self._find_undervalued(undervalued_items["hotel_cities"]),
            "flight_destinations": self._find_undervalued(undervalued_items["flight_destinations"]),
            "days_of_week": self._find_undervalued(days_list),
            "months": self._find_undervalued(months_list)
        }

        # Считаем общее количество
        total = (len(result["hotel_cities"]) +
                 len(result["flight_destinations"]) +
                 len(result["days_of_week"]) +
                 len(result["months"]))

        print(f"\n   📊 Найдено недооцененных:")
        print(f"      Отели: {len(result['hotel_cities'])}")
        print(f"      Авиа: {len(result['flight_destinations'])}")
        print(f"      Дни недели: {len(result['days_of_week'])}")
        print(f"      Месяцы: {len(result['months'])}")
        print(f"      ВСЕГО: {total}")

        return result

    def _find_undervalued(self, items: List[Dict]) -> List[Dict]:
        """Находит недооцененные элементы в списке"""
        if not items:
            return []

        # Вычисляем средние
        avg_price = np.mean([i["avg_price"] for i in items])
        avg_bookings = np.mean([i["booking_count"] for i in items])
        median_price = np.median([i["avg_price"] for i in items])
        median_bookings = np.median([i["booking_count"] for i in items])

        print(f"\n   📊 Статистика для {len(items)} элементов:")
        print(f"      Средняя цена: {avg_price:.0f} руб., медиана: {median_price:.0f} руб.")
        print(f"      Средняя загрузка: {avg_bookings:.0f}, медиана: {median_bookings:.0f}")

        # Ищем аномалии с разными порогами
        undervalued = []

        for item in items:
            # Используем более мягкие критерии
            if item["booking_count"] > median_bookings and item["avg_price"] < avg_price:
                score = (item["booking_count"] / avg_bookings) / (item["avg_price"] / avg_price)

                undervalued.append({
                    "name": item.get("city", item.get("destination", item.get("name", "unknown"))),
                    "avg_price": item["avg_price"],
                    "booking_count": item["booking_count"],
                    "price_vs_avg": f"{(item['avg_price'] / avg_price - 1) * 100:+.1f}%",
                    "load_vs_avg": f"{(item['booking_count'] / avg_bookings - 1) * 100:+.1f}%",
                    "price_vs_median": f"{(item['avg_price'] / median_price - 1) * 100:+.1f}%",
                    "load_vs_median": f"{(item['booking_count'] / median_bookings - 1) * 100:+.1f}%",
                    "score": score
                })

        # Сортируем по score
        undervalued.sort(key=lambda x: x["score"], reverse=True)

        print(f"      Найдено кандидатов: {len(undervalued)}")
        if undervalued:
            print(f"      Топ-3 по потенциалу:")
            for i, item in enumerate(undervalued[:3], 1):
                print(f"         {i}. {item['name']}: цена {item['price_vs_avg']}, загрузка {item['load_vs_avg']}")

        return undervalued[:15]

    def _find_undervalued_dict(self, items: Dict) -> List[Dict]:
        """Находит недооцененные элементы в словаре"""
        if not items:
            return []

        values = list(items.values())
        avg_price = np.mean([v["avg_price"] for v in values])
        avg_bookings = np.mean([v["booking_count"] for v in values])
        median_price = np.median([v["avg_price"] for v in values])
        median_bookings = np.median([v["booking_count"] for v in values])

        undervalued = []

        for name, stats in items.items():
            # Более мягкие критерии
            if stats["booking_count"] > median_bookings and stats["avg_price"] < avg_price:
                score = (stats["booking_count"] / avg_bookings) / (stats["avg_price"] / avg_price)

                undervalued.append({
                    "name": name,
                    "avg_price": stats["avg_price"],
                    "booking_count": stats["booking_count"],
                    "price_vs_avg": f"{(stats['avg_price'] / avg_price - 1) * 100:+.1f}%",
                    "load_vs_avg": f"{(stats['booking_count'] / avg_bookings - 1) * 100:+.1f}%",
                    "price_vs_median": f"{(stats['avg_price'] / median_price - 1) * 100:+.1f}%",
                    "load_vs_median": f"{(stats['booking_count'] / median_bookings - 1) * 100:+.1f}%",
                    "score": score
                })

        undervalued.sort(key=lambda x: x["score"], reverse=True)
        return undervalued[:10] # Конец вставки этой хуйни



    def _generate_reseller_profile(self, clients: List[Dict]) -> str:
        """Генерирует текстовое описание профиля"""
        if not clients:
            return "Недостаточно данных для формирования профиля"

        avg_orders = np.mean([c.get("order_count", 0) for c in clients])
        avg_hypotheses = np.mean([len(c.get("matched_hypotheses", [])) for c in clients])

        # Считаем распределение гипотез
        all_hypotheses = []
        for c in clients:
            all_hypotheses.extend(c.get("matched_hypotheses", []))

        from collections import Counter
        top_hypotheses = Counter(all_hypotheses).most_common(3)

        profile = (
            f"Типичный профиль подозрительного клиента:\n"
            f"• Среднее количество заказов: {avg_orders:.0f}\n"
            f"• В среднем попадает под {avg_hypotheses:.1f} критериев\n"
            f"• Наиболее частые критерии:"
        )

        for hyp, count in top_hypotheses:
            profile += f"\n  - {hyp}: {count} клиентов"

        return profile

    def _generate_answer(self, synthesis: Dict, question: str) -> str:
        """Генерирует человекочитаемый ответ на основе синтеза"""

        if synthesis.get("type") == "gray_reseller_analysis":
            return self._format_gray_reseller_answer(synthesis)
        elif synthesis.get("type") == "undervalued_routes_analysis":
            return self._format_undervalued_routes_answer(synthesis)
        else:
            return f"По вашему вопросу '{question}' выполнен анализ, но результаты требуют дополнительной интерпретации."

    def _format_gray_reseller_answer(self, synthesis: Dict) -> str:
        """Форматирует ответ для серых реселлеров"""

        lines = []
        lines.append("🔍 АНАЛИЗ СЕРЫХ РЕСЕЛЛЕРОВ")
        lines.append("=" * 60)
        lines.append("")

        total = synthesis.get("total_clients_analyzed", 0)
        lines.append(f"📊 Всего проанализировано клиентов: {total}")
        lines.append("")

        risk = synthesis.get("risk_levels", {})
        lines.append("📈 РАСПРЕДЕЛЕНИЕ ПО УРОВНЯМ РИСКА:")
        lines.append(f"   🔴 Высокий риск (≥4 критериев): {risk.get('high_risk', 0)} клиентов")
        lines.append(f"   🟠 Средний риск (3 критерия): {risk.get('medium_risk', 0)} клиентов")
        lines.append(f"   🟡 Низкий риск (2 критерия): {risk.get('low_risk', 0)} клиентов")
        lines.append(f"   🟢 Потенциальный риск (1 критерий): {risk.get('potential_risk', 0)} клиентов")
        lines.append("")

        most_suspicious = synthesis.get("most_suspicious_clients", [])
        if most_suspicious:
            lines.append("⚠️ КЛИЕНТЫ С МАКСИМАЛЬНЫМ РИСКОМ (попали во все гипотезы):")
            for i, client in enumerate(most_suspicious[:3], 1):
                lines.append(f"   {i}. Клиент {client.get('client_rk', 'unknown')}:")
                lines.append(f"      • Заказов: {client.get('order_count', 0)}")
                lines.append(f"      • Критерии: {', '.join(client.get('matched_hypotheses', [])[:3])}")
            lines.append("")

        profile = synthesis.get("typical_profile")
        if profile:
            lines.append("👤 ПОРТРЕТ ТИПИЧНОГО ПОДОЗРИТЕЛЬНОГО КЛИЕНТА:")
            lines.append(f"   {profile}")
            lines.append("")

        lines.append("📊 СТАТИСТИКА ПО ГИПОТЕЗАМ:")
        for hyp_name, stats in synthesis.get("hypothesis_statistics", {}).items():
            lines.append(f"   • {hyp_name}: {stats.get('total_found', 0)} клиентов")

        lines.append("")
        lines.append("💡 ВЫВОД:")
        lines.append("   На основе множественного анализа, клиентов можно считать 'серыми реселлерами',")
        lines.append("   если они попадают минимум в 3 из 5 критериев. Рекомендуется:")
        lines.append("   • Провести дополнительную проверку топ-20 клиентов")
        lines.append("   • Проанализировать их историю заказов вручную")
        lines.append("   • Рассмотреть возможность ограничения промокодов для таких клиентов")

        return "\n".join(lines)

    def _format_undervalued_routes_answer(self, synthesis: Dict) -> str:
        """Форматирует ответ для недооцененных направлений"""

        lines = []
        lines.append("💰 АНАЛИЗ НЕДООЦЕНЕННЫХ НАПРАВЛЕНИЙ")
        lines.append("=" * 60)
        lines.append("")

        total = synthesis.get("total_opportunities", 0)
        lines.append(f"📊 Найдено потенциальных точек роста: {total}")
        lines.append("")

        top = synthesis.get("top_opportunities", [])[:10]
        if top:
            lines.append("🏆 ТОП-10 НАПРАВЛЕНИЙ/ДАТ С ПОТЕНЦИАЛОМ:")
            for i, item in enumerate(top, 1):
                if "avg_price" in item and "booking_count" in item:
                    lines.append(f"   {i}. {item['item']}:")
                    lines.append(f"      • Средняя цена: {item['avg_price']:.0f} руб.")
                    lines.append(f"      • Количество бронирований: {item['booking_count']}")
                else:
                    lines.append(f"   {i}. {item['item']}: {item.get('value', 'N/A')}")
            lines.append("")

        lines.append("💡 РЕКОМЕНДАЦИИ:")
        lines.append("   1. Провести A/B тестирование повышения цен на эти направления")
        lines.append("   2. Проанализировать эластичность спроса")
        lines.append("   3. Рассмотреть возможность динамического ценообразования")

        return "\n".join(lines)


# ============================================================================
# НОВЫЙ ГРАФ ДЛЯ СЛОЖНЫХ ИССЛЕДОВАНИЙ
# ============================================================================

from langgraph.graph import StateGraph, END

@dataclass
class InvestigationState:
    question: str = ""
    question_type: str = ""
    investigation_type: str = ""
    hypotheses: list = field(default_factory=list)
    current_hypothesis_index: int = 0
    hypothesis_results: dict = field(default_factory=dict)
    investigation_complete: bool = False
    analysis: dict = field(default_factory=dict)
    analysis_success: bool = False
    answer: str = ""
    answer_success: bool = False


def should_continue_investigation(state) -> str:
    """Определяет, нужно ли продолжать проверку гипотез"""
    if isinstance(state, dict):
        current_idx = state.get("current_hypothesis_index", 0)
        hypotheses = state.get("hypotheses", [])
    else:
        current_idx = state.current_hypothesis_index
        hypotheses = state.hypotheses

    if current_idx < len(hypotheses):
        return "test_next_hypothesis"
    else:
        return "synthesize_results"


class InvestigationWorkflow:
    """
    Workflow для сложных исследовательских вопросов
    """

    def __init__(self, hypothesis_generator, hypothesis_tester, evidence_synthesizer):
        self.hypothesis_generator = hypothesis_generator
        self.hypothesis_tester = hypothesis_tester
        self.evidence_synthesizer = evidence_synthesizer
        self.graph = self._build_graph()

    def _build_graph(self):
        workflow = StateGraph(InvestigationState)

        # Добавляем узлы
        workflow.add_node("generate_hypotheses", self._wrap(self.hypothesis_generator.execute))
        workflow.add_node("test_hypothesis", self._wrap(self.hypothesis_tester.execute))
        workflow.add_node("synthesize", self._wrap(self.evidence_synthesizer.execute))

        # Добавляем ребра
        workflow.set_entry_point("generate_hypotheses")
        workflow.add_edge("generate_hypotheses", "test_hypothesis")

        workflow.add_conditional_edges(
            "test_hypothesis",
            should_continue_investigation,
            {
                "test_next_hypothesis": "test_hypothesis",
                "synthesize_results": "synthesize"
            }
        )

        workflow.add_edge("synthesize", END)

        return workflow.compile()

    def _wrap(self, func):
        def wrapper(state):
            if isinstance(state, InvestigationState):
                state_dict = asdict(state)
            else:
                state_dict = state
            result_dict = func(state_dict)
            if isinstance(result_dict, InvestigationState):
                return result_dict
            return InvestigationState(**result_dict)
        return wrapper

    def invoke(self, question: str) -> Dict:
        initial_state = InvestigationState(question=question)
        return self.graph.invoke(initial_state)


# ============================================================================
# ОБНОВЛЕННЫЙ ГЛАВНЫЙ WORKFLOW С СУПЕРВИЗОРОМ
# ============================================================================

class EnhancedAnalysisWorkflow:
    """
    Улучшенный workflow с поддержкой сложных исследований
    """

    def __init__(
        self,
        supervisor,
        simple_workflow,
        gray_reseller_workflow,
        undervalued_routes_workflow
    ):
        self.supervisor = supervisor
        self.simple_workflow = simple_workflow
        self.gray_reseller_workflow = gray_reseller_workflow
        self.undervalued_routes_workflow = undervalued_routes_workflow

    def invoke(self, question: str, user_id: str = "default") -> Dict:
        print("\n" + "=" * 80)
        print("🚀 ЗАПУСК УЛУЧШЕННОГО WORKFLOW")
        print("=" * 80)

        # Шаг 1: Определяем тип вопроса
        initial_state = {"question": question, "user_id": user_id}
        supervised_state = self.supervisor.execute(initial_state)

        question_type = supervised_state.get("question_type")

        # Шаг 2: Маршрутизируем в соответствующий workflow
        if question_type == "simple_analytics":
            print("\n📊 Использую простой аналитический workflow")
            result = self.simple_workflow.invoke(question, user_id)

        elif question_type == "complex_investigation":
            print("\n🔬 Использую исследовательский workflow")

            # Определяем подтип исследования
            question_lower = question.lower()
            if "реселл" in question_lower or "мошен" in question_lower or "сер" in question_lower:
                print("   → Обнаружен запрос про серых реселлеров")
                inv_result = self.gray_reseller_workflow.invoke(question)
            elif "дар" in question_lower or "деньг" in question_lower or "недооцен" in question_lower or "направл" in question_lower:
                print("   → Обнаружен запрос про недооцененные направления")
                inv_result = self.undervalued_routes_workflow.invoke(question)
            else:
                print("   → Тип исследования не определен, использую gray_reseller как fallback")
                inv_result = self.gray_reseller_workflow.invoke(question)

            # Форматируем результат в формате, совместимом с оригинальным
            result = self._format_investigation_result(inv_result, question, user_id)

        else:
            # Fallback
            print("⚠️ Тип вопроса не определен, использую простой workflow")
            result = self.simple_workflow.invoke(question, user_id)

        print("\n" + "=" * 80)
        print("✅ WORKFLOW ЗАВЕРШЕН")
        print("=" * 80)

        # Добавляем отладочный вывод
        print(f"\nDEBUG - Результат: {result.get('answer', 'Нет ответа')[:100]}...")

        return result

    def _format_investigation_result(self, inv_result: Dict, question: str, user_id: str) -> Dict:
        """Форматирует результат исследования в стандартный формат"""

        print(f"\n📝 Форматирую результат исследования...")

        # Извлекаем answer из inv_result, если он там есть
        answer = inv_result.get("answer", "")

        # Получаем analysis
        analysis = inv_result.get("analysis", {})

        # Если answer пустой, генерируем из analysis
        if not answer and analysis:
            if analysis.get("type") == "gray_reseller_analysis":
                answer = self._format_gray_reseller_answer(analysis)
            elif analysis.get("type") == "undervalued_routes_analysis":
                # Передаем полный analysis
                answer = self._format_undervalued_routes_answer(analysis)
            else:
                answer = "Анализ выполнен, но результаты требуют дополнительной интерпретации."

        # Если все еще пусто, создаем базовый ответ
        if not answer:
            answer = f"По вашему вопросу '{question}' выполнен анализ. Проверено 5 гипотез."

        # Добавляем отладочный вывод
        print(f"\n🔍 DEBUG - Результат форматирования:")
        print(f"   Тип анализа: {analysis.get('type', 'unknown')}")
        if analysis.get('type') == 'undervalued_routes_analysis':
            hotel = len(analysis.get('hotel_cities', []))
            flight = len(analysis.get('flight_destinations', []))
            print(f"   Найдено отелей: {hotel}, авиа: {flight}, всего: {hotel + flight}")

        result = {
            "question": question,
            "user_id": user_id,
            "question_type": "complex_investigation",
            "investigation_type": inv_result.get("investigation_type", "unknown"),
            "analysis": analysis,
            "answer": answer,
            "answer_success": True
        }

        print(f"   ✓ Ответ сгенерирован, длина: {len(answer)} символов")

        return result

    def _format_gray_reseller_answer(self, analysis: Dict) -> str:
        """Форматирует ответ для серых реселлеров"""

        lines = []
        lines.append("🔍 АНАЛИЗ СЕРЫХ РЕСЕЛЛЕРОВ")
        lines.append("=" * 60)
        lines.append("")

        total = analysis.get("total_clients_analyzed", 0)
        lines.append(f"📊 Всего проанализировано клиентов: {total}")
        lines.append("")

        risk = analysis.get("risk_levels", {})
        lines.append("📈 РАСПРЕДЕЛЕНИЕ ПО УРОВНЯМ РИСКА:")
        lines.append(f"   🔴 Высокий риск (≥4 критериев): {risk.get('high_risk', 0)} клиентов")
        lines.append(f"   🟠 Средний риск (3 критерия): {risk.get('medium_risk', 0)} клиентов")
        lines.append(f"   🟡 Низкий риск (2 критерия): {risk.get('low_risk', 0)} клиентов")
        lines.append(f"   🟢 Потенциальный риск (1 критерий): {risk.get('potential_risk', 0)} клиентов")
        lines.append("")

        most_suspicious = analysis.get("most_suspicious_clients", [])
        if most_suspicious:
            lines.append("⚠️ КЛИЕНТЫ С МАКСИМАЛЬНЫМ РИСКОМ:")
            for i, client in enumerate(most_suspicious[:3], 1):
                lines.append(f"   {i}. Клиент {client.get('client_rk', 'unknown')}:")
                lines.append(f"      • Заказов: {client.get('order_count', 0)}")
            lines.append("")

        profile = analysis.get("typical_profile")
        if profile:
            lines.append("👤 ПОРТРЕТ ТИПИЧНОГО ПОДОЗРИТЕЛЬНОГО КЛИЕНТА:")
            lines.append(f"   {profile}")
            lines.append("")

        lines.append("📊 СТАТИСТИКА ПО ГИПОТЕЗАМ:")
        for hyp_name, stats in analysis.get("hypothesis_statistics", {}).items():
            lines.append(f"   • {hyp_name}: {stats.get('total_found', 0)} клиентов")

        lines.append("")
        lines.append("💡 ВЫВОД:")
        lines.append("   На основе множественного анализа, клиентов можно считать 'серыми реселлерами',")
        lines.append("   если они попадают минимум в 3 из 5 критериев. Рекомендуется:")
        lines.append("   • Провести дополнительную проверку топ-20 клиентов")
        lines.append("   • Проанализировать их историю заказов вручную")
        lines.append("   • Рассмотреть возможность ограничения промокодов для таких клиентов")

        return "\n".join(lines)

    def _format_undervalued_routes_answer(self, analysis: Dict) -> str:
        """Форматирует ответ для недооцененных направлений"""

        # Получаем данные из analysis
        hotel_cities = analysis.get("hotel_cities", [])
        flight_dests = analysis.get("flight_destinations", [])

        lines = []
        lines.append("💰 АНАЛИЗ НЕДООЦЕНЕННЫХ НАПРАВЛЕНИЙ")
        lines.append("=" * 80)
        lines.append("")

        total = len(hotel_cities) + len(flight_dests)
        lines.append(f"📊 Всего найдено потенциальных точек роста: {total}")
        lines.append("")

        if hotel_cities:
            lines.append("🏨 НЕДООЦЕНЕННЫЕ ГОРОДА ОТЕЛЕЙ:")
            for i, item in enumerate(hotel_cities[:10], 1):
                lines.append(f"   {i}. {item['name']}:")
                lines.append(f"      • Средняя цена: {item['avg_price']:.0f} руб. ({item['price_vs_avg']})")
                lines.append(f"      • Количество бронирований: {item['booking_count']} ({item['load_vs_avg']})")
            lines.append("")

        if flight_dests:
            lines.append("✈️ НЕДООЦЕНЕННЫЕ НАПРАВЛЕНИЯ ПЕРЕЛЕТОВ:")
            for i, item in enumerate(flight_dests[:10], 1):
                lines.append(f"   {i}. {item['name']}:")
                lines.append(f"      • Средняя цена: {item['avg_price']:.0f} руб. ({item['price_vs_avg']})")
                lines.append(f"      • Количество бронирований: {item['booking_count']} ({item['load_vs_avg']})")
            lines.append("")

        lines.append("💡 РЕКОМЕНДАЦИИ:")
        lines.append("   1. Для найденных направлений провести A/B тестирование повышения цен на 5-10%")
        lines.append("   2. Отследить эластичность спроса в течение 1-2 месяцев")
        lines.append("   3. Рассмотреть возможность динамического ценообразования")

        return "\n".join(lines)

# УСПЕХ СЕРЫЕ ЧЕЛИКИ

In [124]:
# ============================================================================
# ЗАПУСК УЛУЧШЕННОГО WORKFLOW С ПОДДЕРЖКОЙ СЛОЖНЫХ ВОПРОСОВ
# ============================================================================

print("\n" + "="*80)
print("🚀 ИНИЦИАЛИЗАЦИЯ УЛУЧШЕННОГО АГЕНТА С ПОДДЕРЖКОЙ СЛОЖНЫХ ВОПРОСОВ")
print("="*80)

# СОЗДАЕМ ПРОСТОЙ WORKFLOW (ЭКЗЕМПЛЯР)
print("\n📦 Создаю простой аналитический workflow...")
simple_workflow = init_analysis_workflow(df, schema_dict, rag, llm, cache)

# 1. Создаем супервизор
supervisor = SupervisorNode(llm)

# 2. Создаем генераторы гипотез
gray_reseller_gen = GrayResellerHypothesisGenerator(llm, df, schema_dict)
undervalued_routes_gen = UndervaluedRoutesHypothesisGenerator(llm, df, schema_dict)

# 3. Создаем исполнитель гипотез (используем существующий executor)
hypothesis_tester = HypothesisTester(df, SimpleAnalyticsExecutor(df, schema_dict, cache))

# 4. Создаем синтезатор
evidence_synthesizer = EvidenceSynthesizer(llm)

# 5. Создаем исследовательские workflow
gray_reseller_workflow = InvestigationWorkflow(
    gray_reseller_gen,
    hypothesis_tester,
    evidence_synthesizer
)

undervalued_routes_workflow = InvestigationWorkflow(
    undervalued_routes_gen,
    hypothesis_tester,
    evidence_synthesizer
)

# 6. Создаем улучшенный главный workflow
enhanced_workflow = EnhancedAnalysisWorkflow(
    supervisor=supervisor,
    simple_workflow=simple_workflow,  # ← используем созданный экземпляр
    gray_reseller_workflow=gray_reseller_workflow,
    undervalued_routes_workflow=undervalued_routes_workflow
)

print("\n✅ Улучшенный агент готов к работе!")
print("="*80)

# ============================================================================
# ТЕСТИРОВАНИЕ
# ============================================================================

def test_enhanced_agent():
    """Тестирует улучшенного агента на разных типах вопросов"""

    test_questions = [
        # Простые вопросы (пойдут в старый граф)
        "Как менялась цена авиабилетов в 2024 году?",
        "Максимальный размер скидки в 2024 году",
        "Какая программа лояльности наиболее активно покупает авиабилеты?",

        # Сложные вопросы (пойдут в новые исследовательские графы)
        "Каких клиентов можно считать серыми реселлерами?",
        "Где мы дарим деньги: недооценённые высокозагруженные даты или направления?"
    ]

    for i, question in enumerate(test_questions, 1):
        print(f"\n\n{'='*80}")
        print(f"ТЕСТ {i}: {question}")
        print('='*80)

        result = enhanced_workflow.invoke(question)

        print(f"\n📝 ОТВЕТ:")
        print(result.get('answer', 'Нет ответа'))

        input("\nНажмите Enter для продолжения...")

# ЗАПУСК
if __name__ == "__main__":
    # Быстрый тест на сложном вопросе
    question = "Каких клиентов можно считать серыми реселлерами?"
    print(f"\n🔍 Тестирую вопрос: {question}")

    result = enhanced_workflow.invoke(question)

    print("\n" + "="*80)
    print("📊 РЕЗУЛЬТАТ АНАЛИЗА")
    print("="*80)
    print(result.get('answer', 'Нет ответа'))


🚀 ИНИЦИАЛИЗАЦИЯ УЛУЧШЕННОГО АГЕНТА С ПОДДЕРЖКОЙ СЛОЖНЫХ ВОПРОСОВ

📦 Создаю простой аналитический workflow...
🔧 Инициализирую анализ workflow...
🔧 Строю LangGraph workflow...
  📍 Добавляю узлы...
  🔗 Добавляю рёбра...
  ✓ Граф построен успешно
✓ Workflow инициализирован успешно

✅ Улучшенный агент готов к работе!

🔍 Тестирую вопрос: Каких клиентов можно считать серыми реселлерами?

🚀 ЗАПУСК УЛУЧШЕННОГО WORKFLOW

🤖 SupervisorNode: Определяю тип вопроса...
   Вопрос: Каких клиентов можно считать серыми реселлерами?
   → По LLM: complex_investigation

🔬 Использую исследовательский workflow
   → Обнаружен запрос про серых реселлеров

🕵️ GrayResellerHypothesisGenerator: Генерирую гипотезы...

📊 Найденные колонки из PDF:
   ID клиента: client_rk
   ID заказа: order_rk
   Тип заказа: order_type_cd
   Статус заказа: order_status_cd
   Промокод скидка: promo_code_discount_amt
   Город отеля: hotel_city
   Город вылета: avia_dep_city
   Город прилета: avia_arr_city

📊 Порог частоты для гипотезы:

# Тестинг: продолжение (дарим деньги)

In [127]:
# Тестирование всех вопросов
test_questions = [
    "Как менялась цена авиабилетов в 2024 году?",
    "Максимальный размер скидки в 2024 году",
    "Какая программа лояльности наиболее активно покупает авиабилеты?",
    "Каких клиентов можно считать серыми реселлерами?",
    "Где мы дарим деньги: недооценённые высокозагруженные даты или направления?"
]

for i, q in enumerate(test_questions, 1):
    print(f"\n\n{'='*80}")
    print(f"ТЕСТ {i}: {q}")
    print('='*80)

    result = enhanced_workflow.invoke(q)
    print(f"\n📝 ОТВЕТ:\n{result.get('answer', 'Нет ответа')}")



ТЕСТ 1: Как менялась цена авиабилетов в 2024 году?

🚀 ЗАПУСК УЛУЧШЕННОГО WORKFLOW

🤖 SupervisorNode: Определяю тип вопроса...
   Вопрос: Как менялась цена авиабилетов в 2024 году?
   → По ключевым словам: simple_analytics

📊 Использую простой аналитический workflow

🚀 НАЧИНАЮ ВЫПОЛНЕНИЕ WORKFLOW

🔍 InterpreterNode: Как менялась цена авиабилетов в 2024 году?
  📚 Ищу релевантные описания колонок...
  ✓ Найдено 5 релевантных описаний
  🤖 Анализирую вопрос с LLM...
    ✓ Найден валидный JSON на позиции 30
  ✓ JSON успешно распарсен
  ✓ Добавлен автофильтр: order_type_cd == 'AIR'
  🔎 Проверяю неоднозначности...
  ⚠️ Обнаружены 2 неоднозначность/неоднозначности:
     - Указана цена, но не ясна валюта
     - Несколько вариантов временных колонок

🔀 ROUTING: Обнаружены неоднозначности → CLARIFICATION

📋 ТРЕБУЮТСЯ УТОЧНЕНИЯ

❓ В какой валюте показать цену: рубли (RUB) или евро (EUR)?
   Доступные варианты:
   1. RUB
   2. EUR

   Выбери номер или введи значение:
   → 1
   ✓ Выбран: RUB

❓ Как

In [126]:
import sys

print("\n" + "="*80)
print("📊 ПОЛНЫЙ ОТВЕТ:")
print("="*80)

# Принудительно выводим весь ответ
full_answer = result.get('answer', 'Нет ответа')
print(full_answer)

# Проверяем длину
print(f"\n📏 Длина ответа: {len(full_answer)} символов")

# Если ответ обрезается, пробуем другой способ вывода
if len(full_answer) > 1000:
    print("\n📋 Ответ слишком длинный, разбиваем на части:")
    for i in range(0, len(full_answer), 1000):
        print(f"\n--- Часть {i//1000 + 1} ---")
        print(full_answer[i:i+1000])


📊 ПОЛНЫЙ ОТВЕТ:
💰 АНАЛИЗ НЕДООЦЕНЕННЫХ НАПРАВЛЕНИЙ

📊 Найдено потенциальных точек роста: 0

💡 РЕКОМЕНДАЦИИ:
   1. Провести A/B тестирование повышения цен на эти направления
   2. Проанализировать эластичность спроса
   3. Рассмотреть возможность динамического ценообразования

📏 Длина ответа: 320 символов
